# DCC-GARCH Consensus

Uses a dynamic conditional correlation model to inform portfolio signals.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
DCC-GARCH STRATEGY - FINAL CORRECTED VERSION
=============================================

CRITICAL FIXES IMPLEMENTED:
1. ROLLING OPTIMIZATION: Full grid search per fold with internal train/validation split
   - Hyperparameter optimization for a, b, k, Ï„, and thresholds
   - Validation-based selection to prevent temporal overfitting
   - Proper n_trials tracking per fold

2. ROBUSTNESS METRICS: Complete statistical validation framework
   - WFE (Walk-Forward Efficiency): IS vs OOS performance degradation
   - DSR (Deflated Sharpe Ratio): Multiple testing adjusted significance
   - Hurst Exponent: Mean-reversion vs trend detection
   - CDaR (Conditional Drawdown at Risk): Tail risk measure
   - PSI (Parameter Stability Index): Cross-fold parameter consistency
   - Rolling IC: Signal vs forward returns correlation (proper definition)
   - GRI (Global Robustness Index): Composite score

3. LEVERAGE CONTROL: Dynamic exposure scaling with MAX_LEVERAGE enforcement
   - Pre-trade leverage projection
   - Automatic scaling when combined exposure exceeds limits
   - Works with both compounding and fixed-capital modes

PERIOD: 2017-01-01 to 2025-10-31
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime, timedelta
from typing import Dict, Tuple, List, Optional
from dataclasses import dataclass, field
from scipy.stats import skew, kurtosis, spearmanr, norm
from scipy import stats
from scipy.optimize import minimize
from scipy.special import gamma
from itertools import product

warnings.filterwarnings("ignore")


class StrategyConfig:
    BACKTEST_START_DATE = "2017-01-01"
    BACKTEST_END_DATE = "2025-10-31"
    INITIAL_CAPITAL = 100000.0
    COMPOUNDING_MODE = False
    BENCHMARK_TICKER = "URTH"
    MACRO_ASSETS = ["SPY", "TLT", "SHY", "GLD", "USO", "DBC", "HYG", "LQD", "EEM", "VNQ", "IEF"]
    LONG_TICKER = "QQQ"
    HEDGE_TICKER = "SH"
    TOP_K_MACROS = 6
    MIN_ASSET_CORRELATION = 0.15
    MIN_ASSET_IC = 0.1
    MAX_ASSET_WEIGHT = 0.35
    ABSOLUTE_MIN_CORRELATION = 0.05
    ABSOLUTE_MIN_IC = 0.02
    MIN_EFFECTIVE_OBSERVATIONS = 60
    DCC_A_GRID = [0.01, 0.02, 0.03]
    DCC_B_GRID = [0.9, 0.95, 0.97]
    SIGMOID_K_GRID = [3.0, 5.0, 7.0]
    VOL_TAU_GRID = [0.2, 0.25, 0.3]
    SAFE_ASSETS = ["TLT", "SHY", "GLD"]
    VOL_WINDOW = 20
    MOMENTUM_WINDOW = 21
    MIN_FEATURE_WARMUP_DAYS = 126
    MIN_TRAINING_DAYS = 504
    REFIT_FREQUENCY = 126
    VALIDATE_FRACTION = 0.3
    REBALANCE_ON_REGIME_CHANGE = True
    REBALANCE_FREQUENCY_GUARD = 21
    HYSTERESIS_UP_BUFFER = 0.02
    HYSTERESIS_DOWN_BUFFER = 0.02
    MIN_TRADE_VALUE = 500.0
    MIN_TRADE_SHARES = 1.0
    OPTIMIZE_THRESHOLDS = True
    THRESHOLD_GRID = [0.45, 0.5, 0.55, 0.6, 0.65]
    STRONG_RISK_ON_THRESHOLD = 0.65
    STRONG_RISK_OFF_THRESHOLD = 0.35
    STRONG_RISK_ON_EXPOSURE = 1.2
    MODERATE_RISK_ON_EXPOSURE = 0.9
    NEUTRAL_EXPOSURE = 0.6
    DEFENSIVE_EXPOSURE = 0.3
    STRONG_RISK_OFF_HEDGE = 0.1
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD_QQQ = 0.0003
    BID_ASK_SPREAD_SH = 0.0008
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    STOP_LOSS_ENABLED = False
    STOP_LOSS_VOL_MULTIPLIER = 2.5
    STOP_LOSS_LOOKBACK = 60
    RISK_FREE_RATE = 0.02
    MARGIN_RATE_ANNUAL = 0.03
    MAX_LEVERAGE = 1.5
    MAX_ALLOWED_GAP_DAYS = 7
    MIN_DATA_QUALITY_THRESHOLD = 0.95
    REGIME_COLORS = {
        "Strong Risk-On": "#2ca02c",
        "Moderate Risk-On": "#90EE90",
        "Neutral": "#7f7f7f",
        "Moderate Risk-Off": "#FFA07A",
        "Strong Risk-Off": "#d62728",
    }


class RobustnessMetrics:
    """
    CORRECTED: Comprehensive statistical validation metrics.

    Computes:
    - WFE (Walk-Forward Efficiency): IS vs OOS performance ratio
    - DSR (Deflated Sharpe Ratio): Bailey et al. multiple testing adjustment
    - Hurst Exponent: Detects mean-reversion vs trending behavior
    - CDaR (Conditional Drawdown at Risk): Expected drawdown in worst Î±% cases
    - PSI (Parameter Stability Index): Cross-fold parameter consistency
    - Rolling IC (proper): Correlation between signal and forward returns
    - GRI (Global Robustness Index): Normalized composite score
    """

    @staticmethod
    def calculate_wfe(sharpe_is: float, sharpe_oos: float) -> float:
        """
        Walk-Forward Efficiency: ratio of out-of-sample to in-sample Sharpe.

        Interpretation:
        - WFE > 0.8: Excellent generalization
        - WFE 0.5-0.8: Acceptable performance decay
        - WFE < 0.5: Significant overfitting
        """
        if abs(sharpe_is) < 0.01:
            return 0.0
        return sharpe_oos / sharpe_is

    @staticmethod
    def calculate_deflated_sharpe(
        observed_sharpe: float,
        n_returns: int,
        n_trials: int,
        skewness: float = 0.0,
        kurtosis: float = 3.0,
    ) -> Tuple[float, float]:
        """Deflated Sharpe Ratio with PSR fallback."""
        if n_returns < 10:
            return (0.0, 1.0)
        if n_trials < 1:
            print(f"    âš  n_trials={n_trials} invalid, using n_trials=1 (PSR mode)")
            n_trials = 1
        if abs(observed_sharpe) < 0.01:
            return (0.0, 1.0)
        excess_kurt = kurtosis - 3.0
        try:
            var_sr = (
                1 + (1 - skewness * observed_sharpe) + (excess_kurt - 1) / 4 * observed_sharpe**2
            ) / (n_returns - 1)
            if var_sr <= 0:
                print(f"    âš  var_sr={var_sr:.6f} â‰¤ 0, using PSR fallback")
                var_sr = 1.0 / (n_returns - 1)
            euler_gamma = 0.5772156649
            expected_max_sr = np.sqrt(var_sr) * (
                (1 - euler_gamma) * norm.ppf(1 - 1 / (2 * n_trials))
                + euler_gamma * norm.ppf(1 - 1 / (n_trials * np.e))
            )
            dsr = (observed_sharpe - expected_max_sr) / np.sqrt(var_sr)
            if not np.isfinite(dsr):
                print(f"    âš  DSR={dsr} not finite, using PSR fallback")
                dsr = observed_sharpe / np.sqrt(1.0 / n_returns)
            p_value = max(0.0, min(1.0, 1 - norm.cdf(dsr)))
            return (dsr, p_value)
        except Exception as e:
            print(f"    âš  DSR calculation failed: {str(e)}, using PSR fallback")
            psr = observed_sharpe / np.sqrt(1.0 / n_returns)
            p_value = max(0.0, min(1.0, 1 - norm.cdf(psr)))
            return (psr, p_value)

    @staticmethod
    def calculate_hurst_exponent(returns: pd.Series, max_lag: int = 100) -> float:
        """
        Hurst Exponent via rescaled range (R/S) analysis.

        Interpretation:
        - H < 0.5: Mean-reverting (anti-persistent)
        - H â‰ˆ 0.5: Random walk (geometric Brownian motion)
        - H > 0.5: Trending (persistent, momentum)

        Returns are expected to be mean-reverting for most financial series.
        """
        returns = returns.dropna()
        if len(returns) < max_lag * 2:
            return 0.5
        lags = range(10, min(max_lag, len(returns) // 2))
        tau = []
        for lag in lags:
            n_blocks = len(returns) // lag
            rs_values = []
            for i in range(n_blocks):
                block = returns.iloc[i * lag : (i + 1) * lag].values
                if len(block) < lag:
                    continue
                cumsum = np.cumsum(block - np.mean(block))
                R = np.max(cumsum) - np.min(cumsum)
                S = np.std(block, ddof=1)
                if S > 1e-08:
                    rs_values.append(R / S)
            if len(rs_values) > 0:
                tau.append(np.mean(rs_values))
            else:
                tau.append(np.nan)
        tau = np.array(tau)
        lags = np.array(lags)
        valid = ~np.isnan(tau) & (tau > 0)
        if np.sum(valid) < 5:
            return 0.5
        tau = tau[valid]
        lags = lags[valid]
        log_lags = np.log(lags)
        log_tau = np.log(tau)
        hurst = np.polyfit(log_lags, log_tau, 1)[0]
        return max(0.0, min(1.0, hurst))

    @staticmethod
    def calculate_cdar(returns: pd.Series, alpha: float = 0.05) -> float:
        """CORRECTED: Enhanced CDaR with proper validation."""
        returns = returns.dropna()
        if len(returns) < 50:
            print(f"    âš  CDaR: Insufficient data ({len(returns)} < 50)")
            return 0.0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        if drawdown.min() >= -1e-06:
            print(f"    âš  CDaR: No significant drawdowns detected")
            return 0.0
        sorted_dd = np.sort(drawdown.values)
        var_idx = max(1, int(np.floor(len(sorted_dd) * alpha)))
        cdar = np.mean(sorted_dd[:var_idx])
        if abs(cdar) < 1e-06:
            print(f"    âš  CDaR: Near-zero result ({cdar:.6f}), check data quality")
        return cdar

    @staticmethod
    def calculate_psi(param_history: List[Dict[str, float]], param_names: List[str]) -> float:
        """
        Parameter Stability Index (PSI).

        Measures consistency of optimized parameters across folds.
        Low PSI indicates stable, reliable parameter estimates.
        High PSI suggests parameter instability and potential overfitting.

        PSI = Î£ (p_new - p_old) * ln(p_new / p_old)

        Adapted for continuous parameters by binning.
        """
        if len(param_history) < 2:
            return 0.0
        psi_values = []
        for param_name in param_names:
            values = [fold_params.get(param_name, np.nan) for fold_params in param_history]
            values = [v for v in values if not np.isnan(v)]
            if len(values) < 2:
                continue
            try:
                bins = np.percentile(values, [0, 20, 40, 60, 80, 100])
                bins = np.unique(bins)
                if len(bins) < 3:
                    continue
                mid = len(values) // 2
                old_values = values[:mid]
                new_values = values[mid:]
                old_hist, _ = np.histogram(old_values, bins=bins)
                new_hist, _ = np.histogram(new_values, bins=bins)
                old_prob = old_hist / len(old_values)
                new_prob = new_hist / len(new_values)
                psi_param = 0.0
                for i in range(len(old_prob)):
                    if old_prob[i] > 0 and new_prob[i] > 0:
                        psi_param += (new_prob[i] - old_prob[i]) * np.log(new_prob[i] / old_prob[i])
                psi_values.append(abs(psi_param))
            except Exception:
                continue
        if len(psi_values) == 0:
            return 0.0
        return np.mean(psi_values)

    @staticmethod
    def calculate_rolling_ic_proper(
        signal: pd.Series, forward_returns: pd.Series, window: int = 60
    ) -> float:
        """
        CORRECTED: Rolling Information Coefficient (proper definition).

        Measures rank correlation between signal at T and forward returns at T+1,
        computed in rolling windows to assess consistency.

        This is NOT the IS/OOS Sharpe ratio.

        Returns:
        --------
        mean_ic : float
            Average Spearman correlation across rolling windows
        """
        signal = signal.sort_index()
        forward_returns = forward_returns.sort_index()
        common_idx = signal.index.intersection(forward_returns.index)
        if len(common_idx) < window + 10:
            return 0.0
        signal = signal.loc[common_idx]
        forward_returns = forward_returns.loc[common_idx]
        ic_values = []
        for i in range(window, len(signal)):
            window_signal = signal.iloc[i - window : i]
            window_fwd_ret = forward_returns.iloc[i - window : i]
            valid = ~(window_signal.isna() | window_fwd_ret.isna())
            if valid.sum() < 20:
                continue
            try:
                ic, _ = spearmanr(window_signal[valid], window_fwd_ret[valid])
                if np.isfinite(ic):
                    ic_values.append(ic)
            except Exception:
                continue
        if len(ic_values) == 0:
            return 0.0
        return np.mean(ic_values)

    @staticmethod
    def calculate_gri(
        wfe: float, dsr_pvalue: float, hurst: float, cdar: float, psi: float, rolling_ic: float
    ) -> float:
        """
        Global Robustness Index (GRI).

        Composite score combining all robustness metrics into [0, 1] scale.

        Normalization:
        - WFE: Higher is better (normalize to [0, 1])
        - DSR p-value: Lower is better (1 - p_value)
        - Hurst: Closer to 0.5 is better (mean-reversion expected)
        - CDaR: Less negative is better (normalize)
        - PSI: Lower is better (less instability)
        - Rolling IC: Higher absolute is better

        GRI > 0.65: Robust model
        GRI 0.50-0.65: Acceptable with caution
        GRI < 0.50: High overfitting risk
        """
        wfe_score = np.clip(wfe, 0, 1.5) / 1.5
        dsr_score = 1 - np.clip(dsr_pvalue, 0, 1)
        hurst_score = 1 - 2 * abs(hurst - 0.5)
        cdar_score = np.clip(1 + 2 * cdar, 0, 1)
        psi_score = 1 - np.clip(psi, 0, 0.5) / 0.5
        ic_score = np.clip(abs(rolling_ic), 0, 0.5) / 0.5
        gri = (
            0.25 * wfe_score
            + 0.2 * dsr_score
            + 0.15 * hurst_score
            + 0.15 * cdar_score
            + 0.15 * psi_score
            + 0.1 * ic_score
        )
        return gri


class GARCHStudentT:

    def __init__(
        self, omega_init=1e-06, alpha_init=0.08, beta_init=0.9, df_init=5.0, max_iter=200, tol=1e-06
    ):
        self.omega = omega_init
        self.alpha = alpha_init
        self.beta = beta_init
        self.df = df_init
        self.max_iter = max_iter
        self.tol = tol
        self.converged = False
        self.conditional_volatility = None
        self.standardized_residuals = None
        self.returns = None

    def _student_t_logpdf(self, x, df):
        scale = np.sqrt((df - 2) / df) if df > 2 else 1.0
        x_scaled = x / scale
        log_const = (
            np.log(gamma((df + 1) / 2))
            - np.log(gamma(df / 2))
            - 0.5 * np.log(np.pi * df)
            - np.log(scale)
        )
        log_kernel = -((df + 1) / 2) * np.log(1 + x_scaled**2 / df)
        return log_const + log_kernel

    def _negative_log_likelihood(self, params, returns):
        omega, alpha, beta, df = params
        if omega <= 0 or alpha <= 0 or beta <= 0:
            return 10000000000.0
        if alpha + beta >= 0.9999:
            return 10000000000.0
        if df <= 2.1 or df > 30:
            return 10000000000.0
        n = len(returns)
        sigma2 = np.zeros(n)
        sigma2[0] = np.var(returns)
        for t in range(1, n):
            sigma2[t] = omega + alpha * returns[t - 1] ** 2 + beta * sigma2[t - 1]
            if sigma2[t] <= 0 or not np.isfinite(sigma2[t]):
                return 10000000000.0
        z = returns / np.sqrt(sigma2)
        log_lik = np.sum(self._student_t_logpdf(z, df) - 0.5 * np.log(sigma2))
        return -log_lik if np.isfinite(log_lik) else 10000000000.0

    def fit(self, returns):
        if isinstance(returns, pd.Series):
            self.returns = returns.values
        else:
            self.returns = returns
        self.returns = self.returns[~np.isnan(self.returns)]
        if len(self.returns) < 50:
            raise ValueError(f"Insufficient data: {len(self.returns)} < 50")
        x0 = np.array([self.omega, self.alpha, self.beta, self.df])
        bounds = [(1e-07, 0.0001), (0.01, 0.3), (0.5, 0.95), (2.5, 25.0)]
        result = minimize(
            self._negative_log_likelihood,
            x0=x0,
            args=(self.returns,),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": self.max_iter, "ftol": self.tol},
        )
        if result.success:
            self.converged = True
            self.omega, self.alpha, self.beta, self.df = result.x
        else:
            self.converged = False
        self._compute_conditional_volatility(self.returns)
        return self

    def _compute_conditional_volatility(self, returns):
        n = len(returns)
        sigma2 = np.zeros(n)
        sigma2[0] = np.var(returns)
        for t in range(1, n):
            sigma2[t] = self.omega + self.alpha * returns[t - 1] ** 2 + self.beta * sigma2[t - 1]
        self.conditional_volatility = np.sqrt(sigma2) * np.sqrt(252)
        self.standardized_residuals = returns / (self.conditional_volatility / np.sqrt(252))

    def forecast_one_step(self):
        """Forecast ÏƒÂ²_{T+1} using last observation."""
        if self.conditional_volatility is None:
            raise ValueError("Model not fitted")
        sigma2_T = (self.conditional_volatility[-1] / np.sqrt(252)) ** 2
        epsilon_T = self.returns[-1]
        sigma2_Tp1 = self.omega + self.alpha * epsilon_T**2 + self.beta * sigma2_T
        return np.sqrt(sigma2_Tp1) * np.sqrt(252)


class DCCGarch:

    def __init__(self, a_init=0.02, b_init=0.95, max_iter=100, tol=1e-06):
        self.a = a_init
        self.b = b_init
        self.max_iter = max_iter
        self.tol = tol
        self.converged = False
        self.garch_models = {}
        self.conditional_correlations = None
        self.Q_bar = None
        self.dates = None
        self.Q_last = None
        self.u_last = None

    def fit(self, returns_dict):
        """Unified log returns."""
        assets = list(returns_dict.keys())
        if len(assets) != 2:
            raise ValueError(f"DCC requires 2 assets, got {len(assets)}")
        dfs = []
        for asset in assets:
            df = returns_dict[asset].to_frame(name=asset)
            dfs.append(df)
        aligned = dfs[0].join(dfs[1], how="inner")
        self.dates = aligned.index
        standardized_residuals = {}
        for asset in assets:
            returns = aligned[asset].dropna()
            if len(returns) < 50:
                raise ValueError(f"Insufficient data for {asset}")
            garch = GARCHStudentT()
            garch.fit(returns.values)
            self.garch_models[asset] = garch
            standardized_residuals[asset] = garch.standardized_residuals
        u = np.column_stack([standardized_residuals[asset] for asset in assets])
        n, k = u.shape
        self.Q_bar = np.corrcoef(u.T)
        x0 = np.array([self.a, self.b])
        bounds = [(0.005, 0.15), (0.8, 0.98)]
        result = minimize(
            self._dcc_negative_log_likelihood,
            x0=x0,
            args=(u, self.Q_bar),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": self.max_iter, "ftol": self.tol},
        )
        if result.success:
            self.converged = True
            self.a, self.b = result.x
        else:
            self.converged = False
        self._compute_conditional_correlations(u)
        if self.conditional_correlations is not None:
            self.Q_last = self._Q_from_R(self.conditional_correlations[-1])
            self.u_last = u[-1]
        return self

    def _Q_from_R(self, R):
        """Recover Q from R."""
        d = np.diag(R)
        D_sqrt = np.diag(np.sqrt(d))
        return D_sqrt @ R @ D_sqrt

    def _dcc_negative_log_likelihood(self, params, u, Q_bar):
        a, b = params
        if a <= 0 or b <= 0 or a + b >= 0.9999:
            return 10000000000.0
        n, k = u.shape
        Q = np.zeros((n, k, k))
        Q[0] = Q_bar.copy()
        log_lik = 0.0
        for t in range(1, n):
            u_outer = np.outer(u[t - 1], u[t - 1])
            Q[t] = (1 - a - b) * Q_bar + a * u_outer + b * Q[t - 1]
            Q_diag_inv_sqrt = np.diag(1 / np.sqrt(np.diag(Q[t])))
            R_t = Q_diag_inv_sqrt @ Q[t] @ Q_diag_inv_sqrt
            eigvals = np.linalg.eigvalsh(R_t)
            if np.any(eigvals <= 0):
                return 10000000000.0
            try:
                R_t_inv = np.linalg.inv(R_t)
                log_det_R_t = np.linalg.slogdet(R_t)[1]
                quadratic_form = u[t] @ R_t_inv @ u[t]
                u_norm_sq = u[t] @ u[t]
                log_lik += -0.5 * (log_det_R_t + quadratic_form - u_norm_sq)
            except np.linalg.LinAlgError:
                return 10000000000.0
        return -log_lik if np.isfinite(log_lik) else 10000000000.0

    def _compute_conditional_correlations(self, u):
        n, k = u.shape
        Q = np.zeros((n, k, k))
        R = np.zeros((n, k, k))
        Q[0] = self.Q_bar.copy()
        R[0] = self.Q_bar.copy()
        for t in range(1, n):
            u_outer = np.outer(u[t - 1], u[t - 1])
            Q[t] = (1 - self.a - self.b) * self.Q_bar + self.a * u_outer + self.b * Q[t - 1]
            Q_diag_inv_sqrt = np.diag(1 / np.sqrt(np.diag(Q[t])))
            R[t] = Q_diag_inv_sqrt @ Q[t] @ Q_diag_inv_sqrt
        self.conditional_correlations = R

    def get_correlation_series(self, asset1, asset2):
        """Extract Ï\x81_t as date-indexed Series."""
        assets = list(self.garch_models.keys())
        idx1 = assets.index(asset1)
        idx2 = assets.index(asset2)
        rho_t = self.conditional_correlations[:, idx1, idx2]
        return pd.Series(rho_t, index=self.dates)

    def get_volatility_series(self, asset):
        """Extract Ïƒ_t as date-indexed Series."""
        if asset not in self.garch_models:
            raise ValueError(f"Asset {asset} not in model")
        garch = self.garch_models[asset]
        return pd.Series(garch.conditional_volatility, index=self.dates)

    def predict_one_step(self, new_returns: Dict[str, float]) -> Tuple[float, float]:
        """One-step-ahead DCC prediction for OOS."""
        if self.Q_last is None or self.u_last is None:
            raise ValueError("Model not fitted or no last state available")
        assets = list(self.garch_models.keys())
        u_new = np.zeros(2)
        new_vols = {}
        for i, asset in enumerate(assets):
            if asset not in new_returns:
                raise ValueError(f"Missing return for {asset}")
            r_new = new_returns[asset]
            garch = self.garch_models[asset]
            sigma_Tp1 = garch.forecast_one_step()
            new_vols[asset] = sigma_Tp1
            u_new[i] = r_new / (sigma_Tp1 / np.sqrt(252))
        u_outer = np.outer(self.u_last, self.u_last)
        Q_Tp1 = (1 - self.a - self.b) * self.Q_bar + self.a * u_outer + self.b * self.Q_last
        Q_diag_inv_sqrt = np.diag(1 / np.sqrt(np.diag(Q_Tp1)))
        R_Tp1 = Q_diag_inv_sqrt @ Q_Tp1 @ Q_diag_inv_sqrt
        self.Q_last = Q_Tp1
        self.u_last = u_new
        rho_Tp1 = R_Tp1[0, 1]
        qqq_vol_Tp1 = new_vols[assets[1]]
        return (rho_Tp1, qqq_vol_Tp1)


class DCCConsensusBuilder:

    def __init__(self, k=5.0, tau_sigma=0.25):
        self.k = k
        self.tau_sigma = tau_sigma

    def compute_score_single_date(self, correlations_dict, qqq_vol, weights_dict, safe_assets):
        """Compute score for single date (for OOS prediction)."""
        vol_filter = qqq_vol <= self.tau_sigma
        score = 0.0
        for asset, rho in correlations_dict.items():
            if asset == "QQQ":
                continue
            weight = weights_dict.get(asset, 0.0)
            if asset in safe_assets:
                score -= weight * rho
            elif vol_filter:
                score += weight * rho
        prob_risk_on = 1 / (1 + np.exp(-self.k * score))
        return (score, prob_risk_on, vol_filter)


class FeatureEngine:

    @staticmethod
    def calculate_yang_zhang_volatility(df: pd.DataFrame, window: int) -> pd.Series:
        O, H, L, C = (df["Open"], df["High"], df["Low"], df["Close"])
        log_ho = np.log(H / O)
        log_hc = np.log(H / C)
        log_lo = np.log(L / O)
        log_lc = np.log(L / C)
        log_co = np.log(C / O)
        log_oc = np.log(O / C.shift(1))
        rs_vol = log_ho * log_hc + log_lo * log_lc
        cc_vol = log_co**2
        oc_vol = log_oc**2
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        rs_roll = rs_vol.rolling(window, min_periods=window).mean()
        cc_roll = cc_vol.rolling(window, min_periods=window).mean()
        oc_roll = oc_vol.rolling(window, min_periods=window).mean()
        yz_vol = np.sqrt(oc_roll + k * cc_roll + (1 - k) * rs_roll) * np.sqrt(252)
        return yz_vol

    @staticmethod
    def calculate_momentum(returns: pd.Series, window: int) -> pd.Series:
        total_return = returns.rolling(window, min_periods=window).sum()
        volatility = returns.rolling(window, min_periods=window).std()
        return total_return / (volatility + 1e-06)

    @staticmethod
    def prepare_features(df: pd.DataFrame, asset: str, config: StrategyConfig) -> pd.DataFrame:
        features = df.copy()
        features = features.sort_values("Date").reset_index(drop=True)
        features["log_return"] = np.log(features["Close"] / features["Close"].shift(1))
        features["log_return_lag1"] = features["log_return"].shift(1)
        features["Open_lag1"] = features["Open"].shift(1)
        features["High_lag1"] = features["High"].shift(1)
        features["Low_lag1"] = features["Low"].shift(1)
        features["Close_lag1"] = features["Close"].shift(1)
        lagged_df = pd.DataFrame(
            {
                "Open": features["Open_lag1"],
                "High": features["High_lag1"],
                "Low": features["Low_lag1"],
                "Close": features["Close_lag1"],
            }
        )
        features["volatility_from_lagged"] = FeatureEngine.calculate_yang_zhang_volatility(
            lagged_df, config.VOL_WINDOW
        )
        features["volatility"] = features["volatility_from_lagged"].shift(1)
        features["momentum_from_lagged"] = FeatureEngine.calculate_momentum(
            features["log_return_lag1"], config.MOMENTUM_WINDOW
        )
        features["momentum"] = features["momentum_from_lagged"].shift(1)
        features[f"{asset}_return_lag2"] = features["log_return"].shift(2)
        features[f"{asset}_vol_lag2"] = features["volatility"]
        features[f"{asset}_momentum_lag2"] = features["momentum"]
        return features


class DataFetcher:

    def validate_data_quality(
        self, df: pd.DataFrame, ticker: str, config: StrategyConfig
    ) -> Tuple[bool, str]:
        null_pct = df["Close"].isnull().sum() / len(df)
        if null_pct > 1 - config.MIN_DATA_QUALITY_THRESHOLD:
            return (False, f"Excessive missing: {null_pct:.1%}")
        df = df.sort_values("Date").reset_index(drop=True)
        df["Date"] = pd.to_datetime(df["Date"])
        date_diffs = df["Date"].diff()
        max_gap = date_diffs.max()
        if pd.notna(max_gap) and max_gap.days > config.MAX_ALLOWED_GAP_DAYS:
            return (False, f"Large gap: {max_gap.days} days")
        if len(df) < 252:
            return (False, f"Insufficient data: {len(df)}")
        return (True, "OK")

    def fetch_historical_data(
        self, ticker: str, start_date: str, end_date: str, config: StrategyConfig
    ) -> pd.DataFrame:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data:
            raise ValueError(f"No data for {ticker}")
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        df = df.rename(
            columns={
                "date": "Date",
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            }
        )
        if "Volume" not in df.columns:
            df["Volume"] = 0
        is_valid, error_msg = self.validate_data_quality(df, ticker, config)
        if not is_valid:
            raise ValueError(error_msg)
        print(f"  âœ“ {ticker}: {len(df)} days")
        return df

    def fetch_all_data(self, config: StrategyConfig) -> Dict[str, pd.DataFrame]:
        print(f"\nFetching data...")
        buffer_start = (
            datetime.strptime(config.BACKTEST_START_DATE, "%Y-%m-%d")
            - timedelta(days=config.MIN_TRAINING_DAYS + 200)
        ).strftime("%Y-%m-%d")
        all_tickers = config.MACRO_ASSETS + [
            config.LONG_TICKER,
            config.HEDGE_TICKER,
            config.BENCHMARK_TICKER,
        ]
        seen = set()
        all_tickers = [x for x in all_tickers if not (x in seen or seen.add(x))]
        data_dict = {}
        for ticker in all_tickers:
            try:
                df = self.fetch_historical_data(
                    ticker, buffer_start, config.BACKTEST_END_DATE, config
                )
                data_dict[ticker] = df
            except Exception as e:
                print(f"  âœ— {ticker}: {str(e)}")
                continue
        if len(data_dict) < 4:
            raise ValueError(f"Insufficient data: {len(data_dict)} assets")
        print(f"âœ“ Fetched {len(data_dict)} assets")
        return data_dict


@dataclass
class FoldMetrics:
    """
    CORRECTED: Complete fold metrics with robustness indicators.
    """

    fold_num: int
    train_start_date: pd.Timestamp
    train_end_date: pd.Timestamp
    test_start_date: pd.Timestamp
    test_end_date: pd.Timestamp
    optimal_dcc_a: float
    optimal_dcc_b: float
    optimal_k: float
    optimal_tau_sigma: float
    optimal_risk_on_threshold: float
    optimal_risk_off_threshold: float
    selected_assets: List[str] = field(default_factory=list)
    asset_scores: Dict[str, float] = field(default_factory=dict)
    sharpe_in_sample: float = 0.0
    sharpe_out_of_sample: float = 0.0
    wfe: float = 0.0
    dsr: float = 0.0
    dsr_pvalue: float = 1.0
    hurst_exponent: float = 0.5
    cdar: float = 0.0
    rolling_ic: float = 0.0
    gri: float = 0.0
    n_trials: int = 1


class WalkForwardDCCDetector:

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.dcc_models = {}
        self.fold_metrics = []
        self.param_history = []

    def select_assets_with_filters(
        self, asset_correlations, asset_ic_scores, fold_num: int
    ) -> Tuple[List[str], Dict[str, float]]:
        """
        CORRECTED: Hierarchical filtering with progressive relaxation and controlled fallback.
        """
        discard_log = {}
        valid_assets = list(asset_correlations.keys())
        correlation_thresholds = [
            self.config.MIN_ASSET_CORRELATION,
            self.config.MIN_ASSET_CORRELATION * 0.75,
            self.config.MIN_ASSET_CORRELATION * 0.5,
            self.config.ABSOLUTE_MIN_CORRELATION,
        ]
        ic_thresholds = [
            self.config.MIN_ASSET_IC,
            self.config.MIN_ASSET_IC * 0.75,
            self.config.MIN_ASSET_IC * 0.5,
            self.config.ABSOLUTE_MIN_IC,
        ]
        selected = []
        asset_scores = {}
        for corr_thresh, ic_thresh in zip(correlation_thresholds, ic_thresholds):
            asset_scores = {}
            for asset in valid_assets:
                mean_corr = asset_correlations[asset]
                ic = asset_ic_scores.get(asset, 0.0)
                if abs(mean_corr) < self.config.ABSOLUTE_MIN_CORRELATION:
                    if asset not in discard_log:
                        discard_log[asset] = f"Below HARD floor |Ï\x81|={abs(mean_corr):.3f}"
                    continue
                if abs(ic) < self.config.ABSOLUTE_MIN_IC:
                    if asset not in discard_log:
                        discard_log[asset] = f"Below HARD floor |IC|={abs(ic):.3f}"
                    continue
                if abs(mean_corr) < corr_thresh:
                    if asset not in discard_log:
                        discard_log[asset] = (
                            f"Low correlation (|Ï\x81|={abs(mean_corr):.3f} < {corr_thresh:.3f})"
                        )
                    continue
                if abs(ic) < ic_thresh:
                    if asset not in discard_log:
                        discard_log[asset] = f"Low IC (|IC|={abs(ic):.3f} < {ic_thresh:.3f})"
                    continue
                score = 0.5 * abs(mean_corr) + 0.5 * abs(ic)
                asset_scores[asset] = score
            if len(asset_scores) >= min(3, self.config.TOP_K_MACROS):
                sorted_assets = sorted(asset_scores.items(), key=lambda x: x[1], reverse=True)
                selected = [asset for asset, score in sorted_assets[: self.config.TOP_K_MACROS]]
                if (
                    corr_thresh < self.config.MIN_ASSET_CORRELATION
                    or ic_thresh < self.config.MIN_ASSET_IC
                ):
                    print(f"  âš  Filters relaxed: corrâ‰¥{corr_thresh:.3f}, ICâ‰¥{ic_thresh:.3f}")
                break
        if len(selected) < min(3, self.config.TOP_K_MACROS):
            print(f"  âš  Insufficient assets after hard floors, using controlled fallback")
            composite_scores = []
            for asset in valid_assets:
                corr = abs(asset_correlations.get(asset, 0.0))
                ic = abs(asset_ic_scores.get(asset, 0.0))
                if (
                    corr >= self.config.ABSOLUTE_MIN_CORRELATION
                    and ic >= self.config.ABSOLUTE_MIN_IC
                ):
                    composite_scores.append((asset, 0.5 * corr + 0.5 * ic))
            composite_scores.sort(key=lambda x: x[1], reverse=True)
            selected = [asset for asset, score in composite_scores[: self.config.TOP_K_MACROS]]
            asset_scores = {
                asset: score for asset, score in composite_scores[: self.config.TOP_K_MACROS]
            }
            if len(selected) == 0:
                print(f"  âš âš  EMERGENCY: No assets pass hard floors!")
                selected = valid_assets[: self.config.TOP_K_MACROS]
                asset_scores = {asset: 1.0 for asset in selected}
        selected_scores = {asset: asset_scores[asset] for asset in selected}
        total_score = sum(selected_scores.values())
        if total_score > 0:
            for asset in selected:
                selected_scores[asset] = selected_scores[asset] / total_score
            capped = False
            for asset in selected:
                if selected_scores[asset] > self.config.MAX_ASSET_WEIGHT:
                    selected_scores[asset] = self.config.MAX_ASSET_WEIGHT
                    capped = True
            if capped:
                total = sum(selected_scores.values())
                if total > 0:
                    selected_scores = {k: v / total for k, v in selected_scores.items()}
        else:
            selected_scores = {asset: 1.0 / len(selected) for asset in selected}
        hhi = sum((w**2 for w in selected_scores.values()))
        turnover = 0.0
        if fold_num > 1 and self.fold_metrics:
            prev_weights = self.fold_metrics[-1].asset_scores
            turnover = sum(
                (
                    abs(selected_scores.get(k, 0) - prev_weights.get(k, 0))
                    for k in set(selected_scores) | set(prev_weights)
                )
            )
        if len(discard_log) > 0 and fold_num <= 2:
            print(f"  Discard reasons (first {min(3, len(discard_log))} of {len(discard_log)}):")
            for i, (asset, reason) in enumerate(list(discard_log.items())[:3]):
                print(f"    {asset}: {reason}")
        print(f"  Selected {len(selected)} assets | HHI={hhi:.3f} | Turnover={turnover:.3f}")
        return (selected, selected_scores)

    def optimize_hyperparams_full_grid(self, features_dict, target_data, train_dates, fold_num):
        """Full grid search with train/validation split per fold."""
        print(f"  Optimizing hyperparameters (full grid search)...")
        n_train = len(train_dates)
        n_fit = int(n_train * (1 - self.config.VALIDATE_FRACTION))
        fit_dates = train_dates[:n_fit]
        val_dates = train_dates[n_fit:]
        print(f"    Fit: {len(fit_dates)} days | Validation: {len(val_dates)} days")
        param_grid = list(
            product(
                self.config.DCC_A_GRID,
                self.config.DCC_B_GRID,
                self.config.SIGMOID_K_GRID,
                self.config.VOL_TAU_GRID,
            )
        )
        threshold_grid = self.config.THRESHOLD_GRID if self.config.OPTIMIZE_THRESHOLDS else [0.55]
        n_trials = len(param_grid) * len(threshold_grid)
        print(f"    Testing {n_trials} combinations...")
        best_val_sharpe = -np.inf
        best_params = None
        trial_count = 0
        for a, b, k, tau in param_grid:
            dcc_models_trial = {}
            asset_correlations_trial = {}
            for asset in features_dict.keys():
                if asset == self.config.LONG_TICKER:
                    continue
                asset_df = features_dict[asset]
                asset_fit = asset_df[asset_df["Date"].isin(fit_dates)].copy()
                target_fit = target_data[target_data["Date"].isin(fit_dates)].copy()
                asset_returns = asset_fit.set_index("Date")[f"{asset}_return_lag2"].dropna()
                target_returns = target_fit.set_index("Date")["log_return"].dropna()
                if len(asset_returns) < 50 or len(target_returns) < 50:
                    continue
                returns_dict = {asset: asset_returns, self.config.LONG_TICKER: target_returns}
                try:
                    dcc = DCCGarch(a_init=a, b_init=b)
                    dcc.fit(returns_dict)
                    if not dcc.converged:
                        continue
                    dcc_models_trial[asset] = dcc
                    rho_series = dcc.get_correlation_series(asset, self.config.LONG_TICKER)
                    asset_correlations_trial[asset] = rho_series.mean()
                except Exception:
                    continue
            if len(dcc_models_trial) < 2:
                continue
            asset_ic_scores = {}
            for asset in dcc_models_trial.keys():
                asset_df = features_dict[asset]
                asset_fit = asset_df[asset_df["Date"].isin(fit_dates)].copy()
                target_fit = target_data[target_data["Date"].isin(fit_dates)].copy()
                asset_returns = asset_fit.set_index("Date")[f"{asset}_return_lag2"].dropna()
                target_returns = target_fit.set_index("Date")["log_return"].dropna()
                rho_series = dcc_models_trial[asset].get_correlation_series(
                    asset, self.config.LONG_TICKER
                )
                target_fwd = target_returns.shift(-1).dropna()
                common_idx = rho_series.index.intersection(target_fwd.index)
                if len(common_idx) > 30:
                    ic, _ = spearmanr(rho_series.loc[common_idx], target_fwd.loc[common_idx])
                    asset_ic_scores[asset] = ic
                else:
                    asset_ic_scores[asset] = 0.0
            selected_assets, asset_weights = self.select_assets_with_filters(
                asset_correlations_trial, asset_ic_scores, fold_num
            )
            if len(selected_assets) == 0:
                continue
            for risk_on_threshold in threshold_grid:
                trial_count += 1
                risk_off_threshold = max(0.35, risk_on_threshold - 0.1)
                val_signals = self._generate_validation_signals(
                    dcc_models_trial,
                    selected_assets,
                    asset_weights,
                    val_dates,
                    features_dict,
                    target_data,
                    k,
                    tau,
                    risk_on_threshold,
                    risk_off_threshold,
                )
                if val_signals is None or len(val_signals) < 20:
                    continue
                val_sharpe = self._calculate_validation_sharpe(val_signals, target_data)
                if val_sharpe > best_val_sharpe:
                    best_val_sharpe = val_sharpe
                    best_params = {
                        "a": a,
                        "b": b,
                        "k": k,
                        "tau": tau,
                        "risk_on_threshold": risk_on_threshold,
                        "risk_off_threshold": risk_off_threshold,
                    }
        if best_params is None:
            print(f"    ⚠ No valid combinations, using defaults")
            best_params = {
                "a": 0.02,
                "b": 0.95,
                "k": 5.0,
                "tau": 0.25,
                "risk_on_threshold": 0.55,
                "risk_off_threshold": 0.45,
            }
            n_trials = 1
        else:
            if n_trials == 0:
                n_trials = len(param_grid) * len(threshold_grid)
            print(f"    ✓ Best validation Sharpe: {best_val_sharpe:.2f}")
            print(
                f"      n_trials={n_trials}, a={best_params['a']:.3f}, b={best_params['b']:.3f}, k={best_params['k']:.1f}, τ={best_params['tau']:.2f}"
            )
        return (
            best_params["a"],
            best_params["b"],
            best_params["k"],
            best_params["tau"],
            best_params["risk_on_threshold"],
            best_params["risk_off_threshold"],
            max(1, n_trials),
        )

    def _generate_validation_signals(
        self,
        dcc_models,
        selected_assets,
        asset_weights,
        val_dates,
        features_dict,
        target_data,
        k,
        tau,
        risk_on_threshold,
        risk_off_threshold,
    ):
        """Generate consensus signals on validation set."""
        try:
            signals = []
            builder = DCCConsensusBuilder(k=k, tau_sigma=tau)
            for date in val_dates:
                correlations_pred = {}
                qqq_vol_pred = None
                date_returns = {}
                for asset in selected_assets:
                    if asset not in dcc_models:
                        continue
                    asset_df = features_dict[asset]
                    asset_row = asset_df[asset_df["Date"] == date]
                    if asset_row.empty:
                        continue
                    ret = asset_row.iloc[0][f"{asset}_return_lag2"]
                    if pd.isna(ret):
                        continue
                    date_returns[asset] = ret
                target_row = target_data[target_data["Date"] == date]
                if not target_row.empty:
                    qqq_ret = target_row.iloc[0]["log_return_lag1"]
                    if not pd.isna(qqq_ret):
                        date_returns[self.config.LONG_TICKER] = qqq_ret
                for asset in selected_assets:
                    if asset not in dcc_models:
                        continue
                    if asset not in date_returns:
                        continue
                    if self.config.LONG_TICKER not in date_returns:
                        continue
                    new_returns = {
                        asset: date_returns[asset],
                        self.config.LONG_TICKER: date_returns[self.config.LONG_TICKER],
                    }
                    try:
                        rho_pred, qqq_vol = dcc_models[asset].predict_one_step(new_returns)
                        correlations_pred[asset] = rho_pred
                        if qqq_vol_pred is None:
                            qqq_vol_pred = qqq_vol
                    except Exception:
                        continue
                if len(correlations_pred) == 0 or qqq_vol_pred is None:
                    continue
                score, consensus_prob, vol_filter = builder.compute_score_single_date(
                    correlations_pred, qqq_vol_pred, asset_weights, self.config.SAFE_ASSETS
                )
                if consensus_prob >= self.config.STRONG_RISK_ON_THRESHOLD:
                    exposure = self.config.STRONG_RISK_ON_EXPOSURE
                elif consensus_prob >= risk_on_threshold:
                    exposure = self.config.MODERATE_RISK_ON_EXPOSURE
                elif consensus_prob >= risk_off_threshold:
                    exposure = self.config.NEUTRAL_EXPOSURE
                else:
                    exposure = self.config.DEFENSIVE_EXPOSURE
                signals.append(
                    {"date": date, "consensus_prob": consensus_prob, "target_exposure": exposure}
                )
            return pd.DataFrame(signals) if len(signals) > 0 else None
        except Exception:
            return None

    def _calculate_validation_sharpe(self, val_signals, target_data):
        """CORRECTED: Use proper P&L simulation for validation."""
        if len(val_signals) < 10:
            return -np.inf
        val_signals = val_signals.sort_values("date").reset_index(drop=True)
        portfolio_returns = []
        for i in range(len(val_signals)):
            current_date = val_signals.iloc[i]["date"]
            target_row = target_data[target_data["Date"] == current_date]
            if not target_row.empty:
                ret = target_row.iloc[0]["log_return"]
                if not pd.isna(ret):
                    exposure = val_signals.iloc[i]["target_exposure"]
                    portfolio_returns.append(ret * exposure)
        if len(portfolio_returns) < 10:
            return -np.inf
        returns = pd.Series(portfolio_returns)
        annual_return = returns.mean() * 252
        volatility = returns.std() * np.sqrt(252)
        if volatility > 0.001:
            sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility
        else:
            sharpe = 0.0
        return sharpe

    def walk_forward_predict(
        self, features_dict: Dict[str, pd.DataFrame], target_data: pd.DataFrame
    ) -> pd.DataFrame:
        """Walk-forward with full optimization and one-step-ahead DCC prediction."""
        print("\n" + "=" * 80)
        print("WALK-FORWARD DCC-GARCH DETECTION (FULL OPTIMIZATION)")
        print("=" * 80)
        self.discard_summary = {
            "sync_issues": [],
            "dcc_convergence_failure": [],
            "insufficient_data": [],
            "correlation_filter": [],
            "ic_filter": [],
        }
        common_dates = None
        for asset, df in features_dict.items():
            df = df.dropna(
                subset=[f"{asset}_return_lag2", f"{asset}_vol_lag2", f"{asset}_momentum_lag2"]
            ).copy()
            asset_dates = df["Date"]
            if common_dates is None:
                common_dates = asset_dates
            else:
                common_dates = common_dates[common_dates.isin(asset_dates)]
        target_dates = target_data["Date"]
        common_dates = common_dates[common_dates.isin(target_dates)]
        common_dates = common_dates.sort_values().reset_index(drop=True)
        print(f"Common dates: {len(common_dates)}")
        target_data = target_data.copy()
        target_data["log_return"] = np.log(target_data["Close"] / target_data["Close"].shift(1))
        target_data["log_return_lag1"] = target_data["log_return"].shift(1)
        results = pd.DataFrame({"Date": common_dates})
        results["consensus_probability"] = np.nan
        results["final_regime"] = None
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        last_refit_idx = min_train
        fold_num = 0
        current_dcc_models = {}
        current_weights = {}
        current_k = 5.0
        current_tau = 0.25
        for i in range(min_train, len(common_dates)):
            current_date = common_dates.iloc[i]
            days_since_refit = i - last_refit_idx
            if not current_dcc_models or days_since_refit >= refit_freq:
                fold_num += 1
                train_end = i
                train_start = 0
                print(f"\nFOLD {fold_num}: Refitting at {current_date.date()}")
                train_dates = common_dates[train_start:train_end]
                test_start_idx = train_end
                test_end_idx = min(train_end + refit_freq, len(common_dates))
                (
                    optimal_a,
                    optimal_b,
                    optimal_k,
                    optimal_tau,
                    optimal_risk_on,
                    optimal_risk_off,
                    n_trials,
                ) = self.optimize_hyperparams_full_grid(
                    features_dict, target_data, train_dates, fold_num
                )
                current_k = optimal_k
                current_tau = optimal_tau
                current_dcc_models = {}
                asset_correlations = {}
                asset_ic_scores = {}
                print(f"  Training DCC models with optimal params...")
                for asset in features_dict.keys():
                    if asset == self.config.LONG_TICKER:
                        continue
                    asset_df = features_dict[asset]
                    asset_train = asset_df[asset_df["Date"].isin(train_dates)].copy()
                    target_train = target_data[target_data["Date"].isin(train_dates)].copy()
                    asset_returns = asset_train.set_index("Date")[f"{asset}_return_lag2"].dropna()
                    target_returns = target_train.set_index("Date")["log_return"].dropna()
                    if len(asset_returns) < 50 or len(target_returns) < 50:
                        self.discard_summary["insufficient_data"].append((fold_num, asset))
                        continue
                    returns_dict = {asset: asset_returns, self.config.LONG_TICKER: target_returns}
                    try:
                        dcc = DCCGarch(a_init=optimal_a, b_init=optimal_b)
                        dcc.fit(returns_dict)
                        if not dcc.converged:
                            self.discard_summary["dcc_convergence_failure"].append(
                                (fold_num, asset)
                            )
                            continue
                        current_dcc_models[asset] = dcc
                        rho_series = dcc.get_correlation_series(asset, self.config.LONG_TICKER)
                        mean_corr = rho_series.mean()
                        asset_correlations[asset] = mean_corr
                        target_fwd = target_returns.shift(-1).dropna()
                        common_idx = rho_series.index.intersection(target_fwd.index)
                        if len(common_idx) > 60:
                            ic, _ = spearmanr(
                                rho_series.loc[common_idx], target_fwd.loc[common_idx]
                            )
                            asset_ic_scores[asset] = ic
                        else:
                            asset_ic_scores[asset] = 0.0
                        print(
                            f"    {asset}: Ï\x81Ì„={mean_corr:.3f}, IC={asset_ic_scores[asset]:.3f}"
                        )
                    except Exception as e:
                        self.discard_summary["dcc_convergence_failure"].append((fold_num, asset))
                        print(f"    âœ— {asset}: {str(e)}")
                        continue
                if len(current_dcc_models) < 2:
                    print(f"  âš  Insufficient models")
                    continue
                selected_assets, current_weights = self.select_assets_with_filters(
                    asset_correlations, asset_ic_scores, fold_num
                )
                self.param_history.append(
                    {
                        "a": optimal_a,
                        "b": optimal_b,
                        "k": optimal_k,
                        "tau": optimal_tau,
                        "risk_on": optimal_risk_on,
                        "risk_off": optimal_risk_off,
                    }
                )
                fold_info = FoldMetrics(
                    fold_num=fold_num,
                    train_start_date=train_dates.iloc[0],
                    train_end_date=train_dates.iloc[-1],
                    test_start_date=common_dates.iloc[test_start_idx],
                    test_end_date=common_dates.iloc[test_end_idx - 1],
                    optimal_dcc_a=optimal_a,
                    optimal_dcc_b=optimal_b,
                    optimal_k=optimal_k,
                    optimal_tau_sigma=optimal_tau,
                    optimal_risk_on_threshold=optimal_risk_on,
                    optimal_risk_off_threshold=optimal_risk_off,
                    selected_assets=selected_assets,
                    asset_scores=current_weights,
                    n_trials=n_trials,
                )
                self.fold_metrics.append(fold_info)
                last_refit_idx = i
            if len(current_dcc_models) == 0:
                continue
            current_date_returns = {}
            for asset, dcc in current_dcc_models.items():
                asset_df = features_dict[asset]
                asset_row = asset_df[asset_df["Date"] == current_date]
                if asset_row.empty:
                    continue
                ret = asset_row.iloc[0][f"{asset}_return_lag2"]
                if pd.isna(ret):
                    continue
                current_date_returns[asset] = ret
            target_row = target_data[target_data["Date"] == current_date]
            if not target_row.empty:
                qqq_ret = target_row.iloc[0]["log_return_lag1"]
                if not pd.isna(qqq_ret):
                    current_date_returns[self.config.LONG_TICKER] = qqq_ret
            correlations_pred = {}
            qqq_vol_pred = None
            for asset, dcc in current_dcc_models.items():
                if asset not in current_date_returns:
                    continue
                if self.config.LONG_TICKER not in current_date_returns:
                    continue
                new_returns = {
                    asset: current_date_returns[asset],
                    self.config.LONG_TICKER: current_date_returns[self.config.LONG_TICKER],
                }
                try:
                    rho_pred, qqq_vol = dcc.predict_one_step(new_returns)
                    correlations_pred[asset] = rho_pred
                    if qqq_vol_pred is None:
                        qqq_vol_pred = qqq_vol
                except Exception:
                    continue
            if len(correlations_pred) == 0 or qqq_vol_pred is None:
                continue
            builder = DCCConsensusBuilder(k=current_k, tau_sigma=current_tau)
            score, consensus_prob, vol_filter = builder.compute_score_single_date(
                correlations_pred, qqq_vol_pred, current_weights, self.config.SAFE_ASSETS
            )
            current_fold = self.fold_metrics[-1]
            opt_risk_on = current_fold.optimal_risk_on_threshold
            opt_risk_off = current_fold.optimal_risk_off_threshold
            if consensus_prob >= self.config.STRONG_RISK_ON_THRESHOLD:
                final_regime = "Strong Risk-On"
            elif consensus_prob >= opt_risk_on:
                final_regime = "Moderate Risk-On"
            elif consensus_prob >= opt_risk_off:
                final_regime = "Neutral"
            elif consensus_prob >= self.config.STRONG_RISK_OFF_THRESHOLD:
                final_regime = "Moderate Risk-Off"
            else:
                final_regime = "Strong Risk-Off"
            results.loc[results["Date"] == current_date, "consensus_probability"] = consensus_prob
            results.loc[results["Date"] == current_date, "final_regime"] = final_regime
        print("\n" + "=" * 80)
        print("DETECTION COMPLETE")
        print("=" * 80)
        regime_counts = results["final_regime"].value_counts()
        for regime, count in regime_counts.items():
            print(f"  {regime}: {count} ({count / len(results) * 100:.1f}%)")
        print("\nDISCARD DIAGNOSTIC SUMMARY:")
        print(f"  Insufficient data: {len(self.discard_summary['insufficient_data'])} occurrences")
        print(
            f"  DCC convergence failures: {len(self.discard_summary['dcc_convergence_failure'])} occurrences"
        )
        if len(self.discard_summary["dcc_convergence_failure"]) > 0:
            from collections import Counter

            asset_counts = Counter(
                [asset for fold, asset in self.discard_summary["dcc_convergence_failure"]]
            )
            top_failures = asset_counts.most_common(3)
            print(
                f"    Most frequent failures: {', '.join([f'{asset}({count})' for asset, count in top_failures])}"
            )
        return results

    def calculate_fold_performance(self, portfolio_history, fold_start, fold_end):
        """Calculate basic Sharpe for fold."""
        fold_data = portfolio_history[
            (portfolio_history["Date"] >= fold_start) & (portfolio_history["Date"] <= fold_end)
        ].copy()
        if len(fold_data) < 10:
            return {"sharpe": 0.0, "returns": pd.Series(dtype=float)}
        returns = fold_data["Daily_Returns"].dropna()
        annual_return = returns.mean() * 252
        volatility = returns.std() * np.sqrt(252)
        if volatility > 0.001:
            sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility
        else:
            sharpe = 0.0
        return {"sharpe": sharpe, "returns": returns}

    def update_fold_metrics_with_portfolio_returns(self, portfolio_history, regime_predictions):
        """
        CORRECTED: Compute full robustness metrics per fold with diagnostic logging.

        Metrics:
        - WFE (Walk-Forward Efficiency)
        - DSR (Deflated Sharpe Ratio)
        - Hurst Exponent
        - CDaR (Conditional Drawdown at Risk)
        - PSI (Parameter Stability Index) - computed once across all folds
        - Rolling IC (proper) - signal vs forward returns
        - GRI (Global Robustness Index)
        """
        print("\nCalculating fold metrics with robustness indicators...")
        psi = RobustnessMetrics.calculate_psi(
            self.param_history, ["a", "b", "k", "tau", "risk_on", "risk_off"]
        )
        print(f"  PSI (Parameter Stability Index): {psi:.3f}")
        for fold_idx, fold in enumerate(self.fold_metrics):
            is_metrics = self.calculate_fold_performance(
                portfolio_history, fold.train_start_date, fold.train_end_date
            )
            is_sharpe = is_metrics["sharpe"]
            is_returns = is_metrics["returns"]
            oos_metrics = self.calculate_fold_performance(
                portfolio_history, fold.test_start_date, fold.test_end_date
            )
            oos_sharpe = oos_metrics["sharpe"]
            oos_returns = oos_metrics["returns"]
            print(f"\n  Fold {fold.fold_num} Diagnostics:")
            print(f"    n_trials: {fold.n_trials}")
            print(f"    IS returns: {len(is_returns)} obs")
            print(f"    OOS returns: {len(oos_returns)} obs")
            wfe = RobustnessMetrics.calculate_wfe(is_sharpe, oos_sharpe)
            if len(oos_returns) > 10:
                oos_skew = skew(oos_returns)
                oos_kurt = kurtosis(oos_returns)
                print(f"    OOS skew: {oos_skew:.3f}, OOS kurt: {oos_kurt:.3f}")
                dsr, dsr_pvalue = RobustnessMetrics.calculate_deflated_sharpe(
                    oos_sharpe, len(oos_returns), fold.n_trials, oos_skew, oos_kurt
                )
                if not np.isfinite(dsr) or not np.isfinite(dsr_pvalue):
                    print(f"    âš âš  DSR/p-value not finite! dsr={dsr}, p={dsr_pvalue}")
                    dsr = 0.0
                    dsr_pvalue = 1.0
            else:
                dsr, dsr_pvalue = (0.0, 1.0)
            if len(oos_returns) > 100:
                hurst = RobustnessMetrics.calculate_hurst_exponent(oos_returns)
            else:
                hurst = 0.5
            if len(oos_returns) > 50:
                cdar = RobustnessMetrics.calculate_cdar(oos_returns, alpha=0.05)
            else:
                cdar = 0.0
            fold_regime_data = regime_predictions[
                (regime_predictions["Date"] >= fold.test_start_date)
                & (regime_predictions["Date"] <= fold.test_end_date)
            ].copy()
            if len(fold_regime_data) > 60:
                signal = fold_regime_data.set_index("Date")["consensus_probability"]
                fold_portfolio = portfolio_history[
                    (portfolio_history["Date"] >= fold.test_start_date)
                    & (portfolio_history["Date"] <= fold.test_end_date)
                ].copy()
                forward_returns = fold_portfolio.set_index("Date")["Daily_Returns"].shift(-1)
                rolling_ic = RobustnessMetrics.calculate_rolling_ic_proper(
                    signal, forward_returns, window=60
                )
            else:
                rolling_ic = 0.0
            gri = RobustnessMetrics.calculate_gri(wfe, dsr_pvalue, hurst, cdar, psi, rolling_ic)
            if not np.isfinite(gri):
                print(
                    f"    âš âš  GRI not finite! Inputs: WFE={wfe}, DSR_p={dsr_pvalue}, Hurst={hurst}, CDaR={cdar}, PSI={psi}, IC={rolling_ic}"
                )
                gri = 0.0
            self.fold_metrics[fold_idx].sharpe_in_sample = is_sharpe
            self.fold_metrics[fold_idx].sharpe_out_of_sample = oos_sharpe
            self.fold_metrics[fold_idx].wfe = wfe
            self.fold_metrics[fold_idx].dsr = dsr
            self.fold_metrics[fold_idx].dsr_pvalue = dsr_pvalue
            self.fold_metrics[fold_idx].hurst_exponent = hurst
            self.fold_metrics[fold_idx].cdar = cdar
            self.fold_metrics[fold_idx].rolling_ic = rolling_ic
            self.fold_metrics[fold_idx].gri = gri
            print(f"\n  Fold {fold.fold_num} Results:")
            print(f"    Sharpe: IS={is_sharpe:.2f} | OOS={oos_sharpe:.2f}")
            print(f"    WFE={wfe:.2f} | DSR={dsr:.2f} (p={dsr_pvalue:.3f})")
            print(f"    Hurst={hurst:.2f} | CDaR={cdar:.2%} | IC={rolling_ic:.2f}")
            print(
                f"    GRI={gri:.2f} {('âœ“ ROBUST' if gri > 0.65 else 'âš  REVIEW' if gri > 0.5 else 'âœ— OVERFITTING RISK')}"
            )


class BacktestEngine:

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []
        self.cumulative_pnl = 0.0

    def classify_regime_with_hysteresis(
        self, consensus_prob, last_regime, risk_on_thresh, risk_off_thresh
    ):
        regime_levels = {
            "Strong Risk-On": 4,
            "Moderate Risk-On": 3,
            "Neutral": 2,
            "Moderate Risk-Off": 1,
            "Strong Risk-Off": 0,
        }
        current_level = regime_levels.get(last_regime, 2)
        if consensus_prob >= self.config.STRONG_RISK_ON_THRESHOLD:
            return "Strong Risk-On"
        elif consensus_prob >= risk_on_thresh:
            if (
                current_level < 3
                and consensus_prob >= risk_on_thresh + self.config.HYSTERESIS_UP_BUFFER
            ):
                return "Moderate Risk-On"
            elif current_level >= 3:
                return "Moderate Risk-On"
            return last_regime
        elif consensus_prob >= risk_off_thresh:
            if (
                current_level > 2
                and consensus_prob < risk_off_thresh - self.config.HYSTERESIS_DOWN_BUFFER
            ):
                return "Neutral"
            elif (
                current_level < 2
                and consensus_prob > risk_off_thresh + self.config.HYSTERESIS_UP_BUFFER
            ):
                return "Neutral"
            elif current_level == 2:
                return "Neutral"
            return last_regime
        elif consensus_prob >= self.config.STRONG_RISK_OFF_THRESHOLD:
            if (
                current_level > 1
                and consensus_prob < risk_off_thresh - self.config.HYSTERESIS_DOWN_BUFFER
            ):
                return "Moderate Risk-Off"
            elif current_level <= 1:
                return "Moderate Risk-Off"
            return last_regime
        else:
            return "Strong Risk-Off"

    def calculate_transaction_costs(self, ticker, trade_value):
        commission = max(abs(trade_value) * self.config.COMMISSION_RATE, self.config.MIN_COMMISSION)
        if ticker == self.config.HEDGE_TICKER:
            spread = abs(trade_value) * self.config.BID_ASK_SPREAD_SH
        else:
            spread = abs(trade_value) * self.config.BID_ASK_SPREAD_QQQ
        impact = abs(trade_value) * self.config.MARKET_IMPACT
        return commission + spread + impact

    def run_backtest(self, regime_predictions, data_dict):
        """
        CORRECTED: Enforce MAX_LEVERAGE with dynamic scaling.

        Before each trade, project combined exposure (QQQ + SH).
        If projected leverage > MAX_LEVERAGE, scale down proportionally.
        """
        print("\n" + "=" * 80)
        print("BACKTEST EXECUTION (WITH LEVERAGE CONTROL)")
        print("=" * 80)
        backtest_start = pd.to_datetime(self.config.BACKTEST_START_DATE)
        backtest_end = pd.to_datetime(self.config.BACKTEST_END_DATE)
        backtest_data = (
            regime_predictions[
                (regime_predictions["Date"] >= backtest_start)
                & (regime_predictions["Date"] <= backtest_end)
            ]
            .copy()
            .reset_index(drop=True)
        )
        print(
            f"Period: {backtest_data['Date'].min().date()} to {backtest_data['Date'].max().date()}"
        )
        portfolio_values = []
        daily_returns = []
        daily_pnls = []
        regime_history = []
        consensus_history = []
        capital = self.config.INITIAL_CAPITAL
        qqq_shares = 0.0
        sh_shares = 0.0
        cash = capital
        self.cumulative_pnl = 0.0
        prev_qqq_open = 0.0
        prev_sh_open = 0.0
        prev_qqq_close = 0.0
        prev_sh_close = 0.0
        prev_portfolio_value = capital
        last_rebalance = -self.config.REBALANCE_FREQUENCY_GUARD
        last_regime = "Neutral"
        for day_idx, row in backtest_data.iterrows():
            current_date = row["Date"]
            final_regime = row["final_regime"] if pd.notna(row["final_regime"]) else "Neutral"
            consensus_prob = (
                row["consensus_probability"] if pd.notna(row["consensus_probability"]) else 0.5
            )
            prev_qqq_shares_snapshot = qqq_shares
            prev_sh_shares_snapshot = sh_shares
            qqq_open = None
            sh_open = None
            qqq_close = None
            sh_close = None
            if self.config.LONG_TICKER in data_dict:
                qqq_data = data_dict[self.config.LONG_TICKER]
                qqq_match = qqq_data[qqq_data["Date"] == current_date]
                if not qqq_match.empty:
                    qqq_open = qqq_match.iloc[0]["Open"]
                    qqq_close = qqq_match.iloc[0]["Close"]
            if self.config.HEDGE_TICKER in data_dict:
                sh_data = data_dict[self.config.HEDGE_TICKER]
                sh_match = sh_data[sh_data["Date"] == current_date]
                if not sh_match.empty:
                    sh_open = sh_match.iloc[0]["Open"]
                    sh_close = sh_match.iloc[0]["Close"]
            executed_today = False
            if hasattr(self, "fold_metrics") and self.fold_metrics:
                current_fold = self.fold_metrics[-1]
                opt_risk_on = current_fold.optimal_risk_on_threshold
                opt_risk_off = current_fold.optimal_risk_off_threshold
            else:
                opt_risk_on = 0.55
                opt_risk_off = 0.45
            current_regime = self.classify_regime_with_hysteresis(
                consensus_prob, last_regime, opt_risk_on, opt_risk_off
            )
            regime_changed = current_regime != last_regime
            days_since_rebal = day_idx - last_rebalance
            calendar_guard = days_since_rebal >= self.config.REBALANCE_FREQUENCY_GUARD
            should_rebalance = regime_changed or calendar_guard
            if should_rebalance and day_idx < len(backtest_data) - 1 and (day_idx > 0):
                if qqq_open:
                    prev_date = backtest_data.iloc[day_idx - 1]["Date"]
                    qqq_prev_close = None
                    sh_prev_close = None
                    if self.config.LONG_TICKER in data_dict:
                        qqq_data_prev = data_dict[self.config.LONG_TICKER]
                        qqq_match_prev = qqq_data_prev[qqq_data_prev["Date"] == prev_date]
                        if not qqq_match_prev.empty:
                            qqq_prev_close = qqq_match_prev.iloc[0]["Close"]
                    if self.config.HEDGE_TICKER in data_dict:
                        sh_data_prev = data_dict[self.config.HEDGE_TICKER]
                        sh_match_prev = sh_data_prev[sh_data_prev["Date"] == prev_date]
                        if not sh_match_prev.empty:
                            sh_prev_close = sh_match_prev.iloc[0]["Close"]
                    if qqq_prev_close:
                        if consensus_prob >= self.config.STRONG_RISK_ON_THRESHOLD:
                            target_qqq_alloc = self.config.STRONG_RISK_ON_EXPOSURE
                            target_sh_alloc = 0.0
                        elif consensus_prob >= opt_risk_on:
                            target_qqq_alloc = self.config.MODERATE_RISK_ON_EXPOSURE
                            target_sh_alloc = 0.0
                        elif consensus_prob >= opt_risk_off:
                            target_qqq_alloc = self.config.NEUTRAL_EXPOSURE
                            target_sh_alloc = 0.0
                        elif consensus_prob >= self.config.STRONG_RISK_OFF_THRESHOLD:
                            target_qqq_alloc = self.config.DEFENSIVE_EXPOSURE
                            target_sh_alloc = 0.0
                        else:
                            target_qqq_alloc = self.config.DEFENSIVE_EXPOSURE
                            target_sh_alloc = self.config.STRONG_RISK_OFF_HEDGE
                        if self.config.COMPOUNDING_MODE:
                            reference_capital = prev_portfolio_value
                        else:
                            reference_capital = self.config.INITIAL_CAPITAL
                        raw_qqq_value = reference_capital * target_qqq_alloc
                        raw_sh_value = reference_capital * target_sh_alloc if sh_prev_close else 0.0
                        projected_leverage = (raw_qqq_value + raw_sh_value) / reference_capital
                        if projected_leverage > self.config.MAX_LEVERAGE:
                            scale_factor = self.config.MAX_LEVERAGE / projected_leverage
                            target_qqq_alloc *= scale_factor
                            target_sh_alloc *= scale_factor
                            print(
                                f"    Scaling leverage: {projected_leverage:.2f}x → {self.config.MAX_LEVERAGE:.2f}x"
                            )
                        target_qqq_shares = np.floor(
                            reference_capital * target_qqq_alloc / qqq_prev_close
                        )
                        qqq_shares_to_trade = target_qqq_shares - prev_qqq_shares_snapshot
                        trade_value_abs = abs(qqq_shares_to_trade * qqq_open)
                        if (
                            trade_value_abs >= self.config.MIN_TRADE_VALUE
                            and abs(qqq_shares_to_trade) >= self.config.MIN_TRADE_SHARES
                        ):
                            executed_today = True
                            trade_value = qqq_shares_to_trade * qqq_open
                            cost = self.calculate_transaction_costs(
                                self.config.LONG_TICKER, trade_value
                            )
                            qqq_shares = prev_qqq_shares_snapshot + qqq_shares_to_trade
                            cash -= trade_value + cost
                            self.trades.append(
                                {
                                    "date": current_date,
                                    "ticker": self.config.LONG_TICKER,
                                    "action": "BUY" if qqq_shares_to_trade > 0 else "SELL",
                                    "shares": abs(qqq_shares_to_trade),
                                    "price": qqq_open,
                                    "cost": cost,
                                }
                            )
                        if sh_open and sh_prev_close:
                            target_sh_shares = np.floor(
                                reference_capital * target_sh_alloc / sh_prev_close
                            )
                            sh_shares_to_trade = target_sh_shares - prev_sh_shares_snapshot
                            trade_value_abs = abs(sh_shares_to_trade * sh_open)
                            if (
                                trade_value_abs >= self.config.MIN_TRADE_VALUE
                                and abs(sh_shares_to_trade) >= self.config.MIN_TRADE_SHARES
                            ):
                                executed_today = True
                                trade_value = sh_shares_to_trade * sh_open
                                cost = self.calculate_transaction_costs(
                                    self.config.HEDGE_TICKER, trade_value
                                )
                                sh_shares = prev_sh_shares_snapshot + sh_shares_to_trade
                                cash -= trade_value + cost
                                self.trades.append(
                                    {
                                        "date": current_date,
                                        "ticker": self.config.HEDGE_TICKER,
                                        "action": "BUY" if sh_shares_to_trade > 0 else "SELL",
                                        "shares": abs(sh_shares_to_trade),
                                        "price": sh_open,
                                        "cost": cost,
                                    }
                                )
                        if executed_today:
                            last_rebalance = day_idx
                            last_regime = current_regime
            if day_idx > 0:
                if executed_today:
                    qqq_return = (
                        qqq_open / prev_qqq_open - 1 if qqq_open and prev_qqq_open > 0 else 0
                    )
                    sh_return = sh_open / prev_sh_open - 1 if sh_open and prev_sh_open > 0 else 0
                    qqq_pnl = prev_qqq_shares_snapshot * prev_qqq_open * qqq_return
                    sh_pnl = prev_sh_shares_snapshot * prev_sh_open * sh_return
                    daily_pnl = qqq_pnl + sh_pnl
                else:
                    qqq_return = (
                        qqq_close / prev_qqq_close - 1 if qqq_close and prev_qqq_close > 0 else 0
                    )
                    sh_return = (
                        sh_close / prev_sh_close - 1 if sh_close and prev_sh_close > 0 else 0
                    )
                    qqq_pnl = qqq_shares * prev_qqq_close * qqq_return
                    sh_pnl = sh_shares * prev_sh_close * sh_return
                    daily_pnl = qqq_pnl + sh_pnl
                qqq_exposure = abs(qqq_shares * prev_qqq_close) if prev_qqq_close else 0
                sh_exposure = abs(sh_shares * prev_sh_close) if prev_sh_close else 0
                total_exposure = qqq_exposure + sh_exposure
                reference_capital = (
                    self.config.INITIAL_CAPITAL
                    if not self.config.COMPOUNDING_MODE
                    else prev_portfolio_value
                )
                if total_exposure > reference_capital:
                    borrowed = total_exposure - reference_capital
                    daily_margin_cost = borrowed * (self.config.MARGIN_RATE_ANNUAL / 252)
                    daily_pnl -= daily_margin_cost
                if self.config.COMPOUNDING_MODE:
                    daily_return = (
                        daily_pnl / prev_portfolio_value if prev_portfolio_value > 0 else 0
                    )
                else:
                    daily_return = daily_pnl / self.config.INITIAL_CAPITAL
                daily_pnls.append(daily_pnl)
                self.cumulative_pnl += daily_pnl
            else:
                daily_return = 0.0
                daily_pnl = 0.0
                daily_pnls.append(0.0)
            qqq_value = qqq_shares * qqq_close if qqq_close else 0
            sh_value = sh_shares * sh_close if sh_close else 0
            if self.config.COMPOUNDING_MODE:
                portfolio_value = cash + qqq_value + sh_value
            else:
                portfolio_value = self.config.INITIAL_CAPITAL + self.cumulative_pnl
            portfolio_values.append(portfolio_value)
            daily_returns.append(daily_return)
            regime_history.append(current_regime)
            consensus_history.append(consensus_prob)
            if qqq_open:
                prev_qqq_open = qqq_open
            if sh_open:
                prev_sh_open = sh_open
            if qqq_close:
                prev_qqq_close = qqq_close
            if sh_close:
                prev_sh_close = sh_close
            prev_portfolio_value = portfolio_value
        portfolio_df = pd.DataFrame(
            {
                "Date": backtest_data["Date"],
                "Portfolio_Value": portfolio_values,
                "Daily_Returns": daily_returns,
                "Daily_PnL": daily_pnls,
                "FinalRegime": regime_history,
                "Consensus_Prob": consensus_history,
            }
        )
        final_value = portfolio_df["Portfolio_Value"].iloc[-1]
        returns = portfolio_df["Daily_Returns"].dropna()
        if self.config.COMPOUNDING_MODE:
            total_return = final_value / self.config.INITIAL_CAPITAL - 1
            annual_return = (1 + total_return) ** (252 / len(portfolio_df)) - 1
        else:
            total_return = returns.sum()
            annual_return = total_return * (252 / len(portfolio_df))
        volatility = returns.std() * np.sqrt(252)
        sharpe = (
            (annual_return - self.config.RISK_FREE_RATE) / volatility if volatility > 0.001 else 0
        )
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        metrics = {
            "final_value": final_value,
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_dd,
            "num_trades": len(self.trades),
        }
        print("\n" + "=" * 80)
        print("BACKTEST COMPLETE")
        print(f"Final: ${final_value:,.0f} | Return: {annual_return:.1%} | Sharpe: {sharpe:.2f}")
        print(f"Max DD: {max_dd:.1%} | Trades: {len(self.trades)}")
        print("=" * 80)
        return {"portfolio_history": portfolio_df, "trade_history": self.trades, "metrics": metrics}


class StrategyVisualizer:
    """Complete visualization suite for DCC-GARCH strategy results."""

    def __init__(self, results: Dict, config, detector, benchmark_data: pd.DataFrame = None):
        self.portfolio_history = results["portfolio_history"]
        self.trade_history = results["trade_history"]
        self.metrics = results["metrics"]
        self.config = config
        self.fold_metrics = detector.fold_metrics
        self.benchmark_data = benchmark_data

    def create_full_dashboard(self) -> go.Figure:
        """Create comprehensive dashboard with all visualizations."""
        fig = make_subplots(
            rows=5,
            cols=3,
            row_heights=[0.25, 0.15, 0.2, 0.2, 0.2],
            column_widths=[0.4, 0.3, 0.3],
            subplot_titles=(
                "Portfolio vs Benchmark",
                "Execution Days Overlay",
                "Slippage Distribution",
                "Consensus Probability",
                "Portfolio Leverage",
                "Drawdown",
                "Monthly Returns Heatmap",
                "",
                "",
                "Walk-Forward Performance (Sharpe)",
                "Walk-Forward Performance (WFE)",
                "",
                "",
                "",
                "",
            ),
            specs=[
                [{"type": "scatter"}, {"type": "histogram"}, {"type": "scatter"}],
                [{"type": "scatter"}, {"type": "bar"}, {"type": "scatter"}],
                [{"type": "heatmap", "colspan": 3}, None, None],
                [{"type": "bar"}, {"type": "scatter"}, {"type": "table"}],
                [{"type": "table", "colspan": 3}, None, None],
            ],
            vertical_spacing=0.08,
            horizontal_spacing=0.1,
        )
        self._add_portfolio_vs_benchmark(fig, row=1, col=1)
        self._add_execution_days_overlay(fig, row=1, col=2)
        self._add_slippage_distribution(fig, row=1, col=3)
        self._add_consensus_probability(fig, row=2, col=1)
        self._add_portfolio_leverage(fig, row=2, col=2)
        self._add_drawdown(fig, row=2, col=3)
        self._add_monthly_returns_heatmap(fig, row=3, col=1)
        self._add_walkforward_sharpe(fig, row=4, col=1)
        self._add_walkforward_wfe(fig, row=4, col=2)
        self._add_fold_metrics_table(fig, row=4, col=3)
        self._add_full_metrics_table(fig, row=5, col=1)
        fig.update_layout(
            height=2000,
            title_text=f"DCC-GARCH Strategy - Full Dashboard<br><sub>Period: {self.portfolio_history['Date'].min().date()} to {self.portfolio_history['Date'].max().date()}</sub>",
            title_x=0.5,
            showlegend=True,
            template="plotly_white",
            font=dict(size=10),
        )
        return fig

    def _add_portfolio_vs_benchmark(self, fig, row, col):
        dates = self.portfolio_history["Date"]
        portfolio = self.portfolio_history["Portfolio_Value"]
        portfolio_norm = portfolio / portfolio.iloc[0] * self.config.INITIAL_CAPITAL
        fig.add_trace(
            go.Scatter(
                x=dates, y=portfolio_norm, name="Strategy", line=dict(color="#1f77b4", width=2)
            ),
            row=row,
            col=col,
        )
        if self.benchmark_data is not None:
            benchmark_aligned = self.benchmark_data[self.benchmark_data["Date"].isin(dates)].copy()
            if len(benchmark_aligned) > 0:
                benchmark_aligned = benchmark_aligned.sort_values("Date")
                bench_values = benchmark_aligned["Close"].values
                bench_norm = bench_values / bench_values[0] * self.config.INITIAL_CAPITAL
                fig.add_trace(
                    go.Scatter(
                        x=benchmark_aligned["Date"],
                        y=bench_norm,
                        name="Benchmark (URTH)",
                        line=dict(color="#ff7f0e", width=2, dash="dash"),
                    ),
                    row=row,
                    col=col,
                )
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Value ($)", row=row, col=col)

    def _add_execution_days_overlay(self, fig, row, col):
        if len(self.trade_history) == 0:
            return
        trades_df = pd.DataFrame(self.trade_history)
        trades_df["slippage_pct"] = (
            trades_df["cost"] / (trades_df["shares"] * trades_df["price"]) * 100
        )
        fig.add_trace(
            go.Histogram(
                x=trades_df["slippage_pct"],
                nbinsx=20,
                name="Execution Days",
                marker=dict(color="#9467bd"),
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Slippage (%)", row=row, col=col)
        fig.update_yaxes(title_text="Frequency", row=row, col=col)

    def _add_slippage_distribution(self, fig, row, col):
        if len(self.trade_history) == 0:
            return
        trades_df = pd.DataFrame(self.trade_history)
        trades_df["slippage_pct"] = (
            trades_df["cost"] / (trades_df["shares"] * trades_df["price"]) * 100
        )
        trades_df = trades_df.sort_values("date")
        trades_df["rolling_mean"] = trades_df["slippage_pct"].rolling(20, min_periods=5).mean()
        trades_df["rolling_std"] = trades_df["slippage_pct"].rolling(20, min_periods=5).std()
        fig.add_trace(
            go.Scatter(
                x=trades_df["date"],
                y=trades_df["rolling_mean"],
                name="Mean Slippage",
                line=dict(color="#2ca02c", width=2),
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=trades_df["date"],
                y=trades_df["rolling_mean"] + trades_df["rolling_std"],
                name="+1 Std",
                line=dict(color="gray", width=1, dash="dash"),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=trades_df["date"],
                y=trades_df["rolling_mean"] - trades_df["rolling_std"],
                name="-1 Std",
                line=dict(color="gray", width=1, dash="dash"),
                fill="tonexty",
                fillcolor="rgba(128, 128, 128, 0.2)",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Slippage (%)", row=row, col=col)

    def _add_consensus_probability(self, fig, row, col):
        dates = self.portfolio_history["Date"]
        consensus = self.portfolio_history["Consensus_Prob"]
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=consensus,
                name="Consensus Probability",
                line=dict(color="#8c564b", width=1.5),
                fill="tozeroy",
                fillcolor="rgba(140, 86, 75, 0.2)",
            ),
            row=row,
            col=col,
        )
        fig.add_hline(y=0.5, line_dash="dot", line_color="gray", row=row, col=col)
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Probability", range=[0, 1], row=row, col=col)

    def _add_portfolio_leverage(self, fig, row, col):
        leverage_label = (
            "Neutral (Fixed Notional)" if not self.config.COMPOUNDING_MODE else "Variable"
        )
        leverage_value = 0 if not self.config.COMPOUNDING_MODE else 1
        fig.add_trace(
            go.Bar(
                x=[leverage_label], y=[leverage_value], marker=dict(color="gray"), showlegend=False
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Leverage Type", row=row, col=col)
        fig.update_yaxes(title_text="Mode", row=row, col=col, visible=False)

    def _add_drawdown(self, fig, row, col):
        returns = self.portfolio_history["Daily_Returns"].dropna()
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        dates = self.portfolio_history["Date"].iloc[1:]
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=drawdown * 100,
                name="Drawdown",
                line=dict(color="#d62728", width=1),
                fill="tozeroy",
                fillcolor="rgba(214, 39, 40, 0.3)",
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Date", row=row, col=col)
        fig.update_yaxes(title_text="Drawdown (%)", row=row, col=col)

    def _add_monthly_returns_heatmap(self, fig, row, col):
        portfolio = self.portfolio_history.copy()
        portfolio["Year"] = portfolio["Date"].dt.year
        portfolio["Month"] = portfolio["Date"].dt.month
        monthly_returns = []
        for year in sorted(portfolio["Year"].unique()):
            year_data = []
            for month in range(1, 13):
                month_data = portfolio[
                    (portfolio["Year"] == year) & (portfolio["Month"] == month)
                ].copy()
                if len(month_data) > 0:
                    start_value = month_data["Portfolio_Value"].iloc[0]
                    end_value = month_data["Portfolio_Value"].iloc[-1]
                    monthly_return = (end_value / start_value - 1) * 100
                    year_data.append(monthly_return)
                else:
                    year_data.append(np.nan)
            monthly_returns.append(year_data)
        years = sorted(portfolio["Year"].unique())
        months = [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ]
        z_data = np.array(monthly_returns)
        text_data = []
        for row_data in z_data:
            row_text = ["" if np.isnan(val) else f"{val:.1f}" for val in row_data]
            text_data.append(row_text)
        fig.add_trace(
            go.Heatmap(
                z=z_data,
                x=months,
                y=[str(y) for y in years],
                text=text_data,
                texttemplate="%{text}",
                textfont=dict(size=10),
                colorscale="RdYlGn",
                zmid=0,
                colorbar=dict(title="Return (%)", x=1.15),
                hovertemplate="%{y} %{x}: %{z:.2f}%<extra></extra>",
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Month", row=row, col=col)
        fig.update_yaxes(title_text="Year", row=row, col=col)

    def _add_walkforward_sharpe(self, fig, row, col):
        if len(self.fold_metrics) == 0:
            return
        fold_nums = [f.fold_num for f in self.fold_metrics]
        is_sharpe = [f.sharpe_in_sample for f in self.fold_metrics]
        oos_sharpe = [f.sharpe_out_of_sample for f in self.fold_metrics]
        fig.add_trace(
            go.Bar(x=fold_nums, y=is_sharpe, name="In-Sample", marker=dict(color="#2ca02c")),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Bar(x=fold_nums, y=oos_sharpe, name="Out-of-Sample", marker=dict(color="#17becf")),
            row=row,
            col=col,
        )
        fig.add_hline(y=0, line_dash="dash", line_color="gray", row=row, col=col)
        fig.update_xaxes(title_text="Fold", row=row, col=col)
        fig.update_yaxes(title_text="Sharpe Ratio", row=row, col=col)

    def _add_walkforward_wfe(self, fig, row, col):
        if len(self.fold_metrics) == 0:
            return
        fold_nums = [f.fold_num for f in self.fold_metrics]
        wfe_values = [f.wfe for f in self.fold_metrics]
        fig.add_trace(
            go.Scatter(
                x=fold_nums,
                y=wfe_values,
                name="WFE",
                line=dict(color="#1f77b4", width=2),
                mode="lines+markers",
            ),
            row=row,
            col=col,
        )
        fig.add_hline(
            y=0.8,
            line_dash="dash",
            line_color="green",
            annotation_text="Good (0.8)",
            row=row,
            col=col,
        )
        fig.add_hline(
            y=0.5,
            line_dash="dash",
            line_color="red",
            annotation_text="Overfitting (0.5)",
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Fold", row=row, col=col)
        fig.update_yaxes(title_text="WFE", row=row, col=col)

    def _add_fold_metrics_table(self, fig, row, col):
        if len(self.fold_metrics) == 0:
            return
        avg_is_sharpe = np.mean([f.sharpe_in_sample for f in self.fold_metrics])
        avg_oos_sharpe = np.mean([f.sharpe_out_of_sample for f in self.fold_metrics])
        avg_wfe = np.mean([f.wfe for f in self.fold_metrics])
        avg_gri = np.mean([f.gri for f in self.fold_metrics])
        robust_count = sum((1 for f in self.fold_metrics if f.gri > 0.65))
        header = ["Metric", "Value"]
        cells = [
            ["Folds", "Avg IS Sharpe", "Avg OOS Sharpe", "Avg WFE", "Avg GRI", "Robust Folds"],
            [
                str(len(self.fold_metrics)),
                f"{avg_is_sharpe:.2f}",
                f"{avg_oos_sharpe:.2f}",
                f"{avg_wfe:.2f}",
                f"{avg_gri:.2f}",
                f"{robust_count}/{len(self.fold_metrics)}",
            ],
        ]
        fig.add_trace(
            go.Table(
                header=dict(values=header, fill_color="paleturquoise", align="left"),
                cells=dict(values=cells, fill_color="lavender", align="left"),
            ),
            row=row,
            col=col,
        )

    def _add_full_metrics_table(self, fig, row, col):
        returns = self.portfolio_history["Daily_Returns"].dropna()
        annual_return = self.metrics["annual_return"]
        max_dd = abs(self.metrics["max_drawdown"])
        calmar = annual_return / max_dd if max_dd > 0.01 else 0
        downside_returns = returns[returns < 0]
        downside_std = downside_returns.std() * np.sqrt(252)
        sortino = (
            (annual_return - self.config.RISK_FREE_RATE) / downside_std
            if downside_std > 0.001
            else 0
        )
        win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
        header = ["Metric", "Value"]
        cells = [
            [
                "Mode",
                "Period",
                "",
                "STRATEGY",
                "Final Value",
                "Total Return",
                "Annual Return",
                "Volatility",
                "Sharpe Ratio",
                "Sortino Ratio",
                "Calmar Ratio",
                "Max Drawdown",
                "Win Rate",
                "Total Trades",
            ],
            [
                "Fixed Notional" if not self.config.COMPOUNDING_MODE else "Compounding",
                f"{self.portfolio_history['Date'].min().date()} to {self.portfolio_history['Date'].max().date()}",
                "",
                "",
                f"${self.metrics['final_value']:,.0f}",
                f"{self.metrics['total_return']:.2%}",
                f"{self.metrics['annual_return']:.2%}",
                f"{self.metrics['volatility']:.2%}",
                f"{self.metrics['sharpe_ratio']:.2f}",
                f"{sortino:.2f}",
                f"{calmar:.2f}",
                f"{self.metrics['max_drawdown']:.2%}",
                f"{win_rate:.2%}",
                str(self.metrics["num_trades"]),
            ],
        ]
        fig.add_trace(
            go.Table(
                header=dict(
                    values=header,
                    fill_color="lightblue",
                    align="left",
                    font=dict(size=12, color="black"),
                ),
                cells=dict(values=cells, fill_color="white", align="left", font=dict(size=11)),
            ),
            row=row,
            col=col,
        )


class MonteCarloSimulator:
    """Monte Carlo simulation for forward portfolio projections."""

    def __init__(
        self,
        portfolio_history: pd.DataFrame,
        n_simulations: int = 10000,
        forward_days: int = 63,
        initial_capital: float = None,
    ):
        self.portfolio_history = portfolio_history
        self.n_simulations = n_simulations
        self.forward_days = forward_days
        self.initial_capital = (
            initial_capital
            if initial_capital is not None
            else portfolio_history["Portfolio_Value"].iloc[-1]
        )
        self.returns = portfolio_history["Daily_Returns"].dropna()
        self._fit_return_distribution()
        self.simulated_paths = None
        self.final_values = None
        self.final_returns = None
        self.max_drawdowns = None

    def _fit_return_distribution(self):
        self.mean_return = self.returns.mean()
        self.std_return = self.returns.std()
        self.skewness = stats.skew(self.returns)
        self.kurtosis_excess = stats.kurtosis(self.returns)
        print(f"Historical return characteristics:")
        print(f"  Mean: {self.mean_return:.6f} ({self.mean_return * 252:.2%} annualized)")
        print(f"  Std: {self.std_return:.6f} ({self.std_return * np.sqrt(252):.2%} annualized)")
        print(f"  Skewness: {self.skewness:.4f}")
        print(f"  Excess Kurtosis: {self.kurtosis_excess:.4f}")

    def run_simulation(self):
        print(
            f"\nRunning Monte Carlo: {self.n_simulations:,} simulations x {self.forward_days} days..."
        )
        self.simulated_paths = np.zeros((self.n_simulations, self.forward_days + 1))
        self.simulated_paths[:, 0] = self.initial_capital
        returns_array = self.returns.values
        for sim in range(self.n_simulations):
            simulated_returns = np.random.choice(
                returns_array, size=self.forward_days, replace=True
            )
            for day in range(self.forward_days):
                self.simulated_paths[sim, day + 1] = self.simulated_paths[sim, day] * (
                    1 + simulated_returns[day]
                )
        self.final_values = self.simulated_paths[:, -1]
        self.final_returns = (self.final_values / self.initial_capital - 1) * 100
        self.max_drawdowns = self._calculate_max_drawdowns()
        print(f"âœ“ Simulation complete")

    def _calculate_max_drawdowns(self) -> np.ndarray:
        max_dds = np.zeros(self.n_simulations)
        for sim in range(self.n_simulations):
            path = self.simulated_paths[sim, :]
            running_max = np.maximum.accumulate(path)
            drawdown = (path - running_max) / running_max
            max_dds[sim] = drawdown.min() * 100
        return max_dds

    def calculate_statistics(self) -> Dict:
        return {
            "mean_final_value": np.mean(self.final_values),
            "median_final_value": np.median(self.final_values),
            "std_final_value": np.std(self.final_values),
            "mean_return": np.mean(self.final_returns),
            "median_return": np.median(self.final_returns),
            "std_return": np.std(self.final_returns),
            "p05_value": np.percentile(self.final_values, 5),
            "p10_value": np.percentile(self.final_values, 10),
            "p25_value": np.percentile(self.final_values, 25),
            "p50_value": np.percentile(self.final_values, 50),
            "p75_value": np.percentile(self.final_values, 75),
            "p90_value": np.percentile(self.final_values, 90),
            "p95_value": np.percentile(self.final_values, 95),
            "p05_return": np.percentile(self.final_returns, 5),
            "p10_return": np.percentile(self.final_returns, 10),
            "p25_return": np.percentile(self.final_returns, 25),
            "p50_return": np.percentile(self.final_returns, 50),
            "p75_return": np.percentile(self.final_returns, 75),
            "p90_return": np.percentile(self.final_returns, 90),
            "p95_return": np.percentile(self.final_returns, 95),
            "var_95": np.percentile(self.final_returns, 5),
            "cvar_95": np.mean(
                self.final_returns[self.final_returns <= np.percentile(self.final_returns, 5)]
            ),
            "var_99": np.percentile(self.final_returns, 1),
            "cvar_99": np.mean(
                self.final_returns[self.final_returns <= np.percentile(self.final_returns, 1)]
            ),
            "mean_max_dd": np.mean(self.max_drawdowns),
            "median_max_dd": np.median(self.max_drawdowns),
            "prob_profit": np.sum(self.final_returns > 0) / self.n_simulations * 100,
            "prob_loss": np.sum(self.final_returns < 0) / self.n_simulations * 100,
            "prob_loss_10pct": np.sum(self.final_returns < -10) / self.n_simulations * 100,
            "prob_gain_20pct": np.sum(self.final_returns > 20) / self.n_simulations * 100,
        }

    def create_visualization(self) -> go.Figure:
        if self.simulated_paths is None:
            raise ValueError("Run simulation first")
        stats = self.calculate_statistics()
        fig = make_subplots(
            rows=3,
            cols=3,
            row_heights=[0.4, 0.3, 0.3],
            column_widths=[0.33, 0.33, 0.34],
            subplot_titles=(
                "",
                "",
                "",
                "Final Value Distribution",
                "Return Distribution",
                "Value at Risk (VaR) & CVaR",
                "Maximum Drawdown Distribution",
                "Probability of Outcomes",
                "",
            ),
            specs=[
                [{"type": "scatter", "colspan": 3}, None, None],
                [{"type": "histogram"}, {"type": "histogram"}, {"type": "bar"}],
                [{"type": "histogram"}, {"type": "bar"}, {"type": "table"}],
            ],
            vertical_spacing=0.1,
            horizontal_spacing=0.1,
        )
        self._add_fan_chart(fig, stats, row=1, col=1)
        self._add_final_value_distribution(fig, stats, row=2, col=1)
        self._add_return_distribution(fig, stats, row=2, col=2)
        self._add_var_cvar_chart(fig, stats, row=2, col=3)
        self._add_max_drawdown_distribution(fig, stats, row=3, col=1)
        self._add_probability_outcomes(fig, stats, row=3, col=2)
        self._add_percentile_table(fig, stats, row=3, col=3)
        fig.update_layout(
            height=1400,
            title_text=f"Monte Carlo Simulation - {self.forward_days} Trading Days Forward<br><sub>{self.n_simulations:,} simulations | Initial: ${self.initial_capital:,.0f}</sub>",
            title_x=0.5,
            showlegend=True,
            template="plotly_white",
            font=dict(size=10),
        )
        return fig

    def _add_fan_chart(self, fig, stats, row, col):
        days = np.arange(self.forward_days + 1)
        p05 = np.percentile(self.simulated_paths, 5, axis=0)
        p50 = np.percentile(self.simulated_paths, 50, axis=0)
        p95 = np.percentile(self.simulated_paths, 95, axis=0)
        mean_path = np.mean(self.simulated_paths, axis=0)
        fig.add_trace(
            go.Scatter(x=days, y=p05, name="P5", line=dict(color="#8B0000", width=2), mode="lines"),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=days, y=p50, name="P50", line=dict(color="#228B22", width=2), mode="lines"
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=days, y=p95, name="P95", line=dict(color="#FF8C00", width=2), mode="lines"
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=days,
                y=mean_path,
                name="Mean",
                line=dict(color="#000080", width=2, dash="dash"),
                mode="lines",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=days,
                y=[self.initial_capital] * len(days),
                name="Initial",
                line=dict(color="gray", width=1, dash="dot"),
                mode="lines",
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Trading Days", row=row, col=col)
        fig.update_yaxes(title_text="Portfolio Value ($)", row=row, col=col)
        fig.add_annotation(
            text="Simulated Portfolio Paths (Fan Chart)",
            xref="paper",
            yref="paper",
            x=0.5,
            y=0.98,
            xanchor="center",
            yanchor="top",
            showarrow=False,
            font=dict(size=14, color="black"),
        )

    def _add_final_value_distribution(self, fig, stats, row, col):
        fig.add_trace(
            go.Histogram(
                x=self.final_values,
                nbinsx=50,
                marker=dict(color="rgba(100, 149, 237, 0.7)", line=dict(color="black", width=1)),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_vline(
            x=self.initial_capital,
            line_dash="dash",
            line_color="red",
            annotation_text="Initial",
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["mean_final_value"],
            line_dash="dash",
            line_color="black",
            annotation_text="Mean",
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["median_final_value"],
            line_dash="dot",
            line_color="green",
            annotation_text="Median",
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["p05_value"],
            line_dash="dot",
            line_color="red",
            annotation_text="VaR 95%",
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Portfolio Value ($)", row=row, col=col)
        fig.update_yaxes(title_text="Density", row=row, col=col)

    def _add_return_distribution(self, fig, stats, row, col):
        fig.add_trace(
            go.Histogram(
                x=self.final_returns,
                nbinsx=50,
                marker=dict(color="rgba(34, 139, 34, 0.7)", line=dict(color="black", width=1)),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["mean_return"],
            line_dash="dash",
            line_color="black",
            annotation_text=f"Mean: {stats['mean_return']:.1f}%",
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["median_return"],
            line_dash="dot",
            line_color="green",
            annotation_text=f"Median: {stats['median_return']:.1f}%",
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Return (%)", row=row, col=col)
        fig.update_yaxes(title_text="Density", row=row, col=col)

    def _add_var_cvar_chart(self, fig, stats, row, col):
        categories = ["VaR 95%", "CVaR 95%", "VaR 99%", "CVaR 99%"]
        values = [stats["var_95"], stats["cvar_95"], stats["var_99"], stats["cvar_99"]]
        colors = ["#FF8C00", "#8B0000", "#FF8C00", "#8B0000"]
        fig.add_trace(
            go.Bar(
                x=categories,
                y=values,
                marker=dict(color=colors),
                text=[f"{v:.1f}%" for v in values],
                textposition="inside",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Percentile Summary", row=row, col=col)
        fig.update_yaxes(title_text="Loss (%)", row=row, col=col)

    def _add_max_drawdown_distribution(self, fig, stats, row, col):
        fig.add_trace(
            go.Histogram(
                x=self.max_drawdowns,
                nbinsx=50,
                marker=dict(color="rgba(139, 0, 0, 0.7)", line=dict(color="black", width=1)),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["mean_max_dd"],
            line_dash="dash",
            line_color="black",
            annotation_text=f"Mean: {stats['mean_max_dd']:.1f}%",
            row=row,
            col=col,
        )
        fig.add_vline(
            x=stats["median_max_dd"],
            line_dash="dot",
            line_color="green",
            annotation_text=f"Median: {stats['median_max_dd']:.1f}%",
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="Max Drawdown (%)", row=row, col=col)
        fig.update_yaxes(title_text="Density", row=row, col=col)

    def _add_probability_outcomes(self, fig, stats, row, col):
        categories = ["Profit", "Loss", "Loss > 10%", "Gain > 20%"]
        probabilities = [
            stats["prob_profit"],
            stats["prob_loss"],
            stats["prob_loss_10pct"],
            stats["prob_gain_20pct"],
        ]
        colors = ["#228B22", "#8B0000", "#8B0000", "#228B22"]
        fig.add_trace(
            go.Bar(
                x=categories,
                y=probabilities,
                marker=dict(color=colors),
                text=[f"{p:.1f}%" for p in probabilities],
                textposition="outside",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="", row=row, col=col)
        fig.update_yaxes(title_text="Probability (%)", row=row, col=col)

    def _add_percentile_table(self, fig, stats, row, col):
        percentiles = ["P5", "P10", "P25", "P50", "P75", "P90", "P95"]
        values = [
            f"${stats[f'p{p}_value']:,.0f}" for p in ["05", "10", "25", "50", "75", "90", "95"]
        ]
        returns = [
            f"{stats[f'p{p}_return']:.1f}%" for p in ["05", "10", "25", "50", "75", "90", "95"]
        ]
        fill_colors = []
        for ret_str in returns:
            ret_val = float(ret_str.strip("%"))
            if ret_val < -10:
                fill_colors.append("#FFB6C1")
            elif ret_val < 0:
                fill_colors.append("#FFE4E1")
            elif ret_val < 5:
                fill_colors.append("#F0FFF0")
            else:
                fill_colors.append("#90EE90")
        fig.add_trace(
            go.Table(
                header=dict(
                    values=["<b>Percentile</b>", "<b>Value</b>", "<b>Return</b>"],
                    fill_color="#90EE90",
                    align="left",
                    font=dict(size=11, color="black"),
                ),
                cells=dict(
                    values=[percentiles, values, returns],
                    fill_color=[["white"] * 7, ["white"] * 7, fill_colors],
                    align="left",
                    font=dict(size=10),
                ),
            ),
            row=row,
            col=col,
        )


def main():
    print("=" * 80)
    print("DCC-GARCH STRATEGY - FINAL CORRECTED VERSION")
    print("=" * 80)
    print("âœ“ Full rolling optimization with train/validation split")
    print("âœ“ Complete robustness metrics (WFE, DSR, Hurst, CDaR, PSI, IC, GRI)")
    print("âœ“ Dynamic leverage control with MAX_LEVERAGE enforcement")
    print("=" * 80)
    config = StrategyConfig()
    print("\nSTEP 1: DATA FETCHING")
    fetcher = DataFetcher()
    data_dict = fetcher.fetch_all_data(config)
    print("\nSTEP 2: FEATURE ENGINEERING")
    features_dict = {}
    for asset in config.MACRO_ASSETS:
        if asset in data_dict:
            features_df = FeatureEngine.prepare_features(data_dict[asset], asset, config)
            features_dict[asset] = features_df
    print("\nSTEP 3: DCC-GARCH DETECTION")
    target_data = data_dict[config.LONG_TICKER]
    detector = WalkForwardDCCDetector(config)
    regime_predictions = detector.walk_forward_predict(features_dict, target_data)
    print("\nSTEP 4: BACKTESTING")
    engine = BacktestEngine(config)
    engine.fold_metrics = detector.fold_metrics
    results = engine.run_backtest(regime_predictions, data_dict)
    print("\nSTEP 5: ROBUSTNESS ANALYSIS")
    detector.update_fold_metrics_with_portfolio_returns(
        results["portfolio_history"], regime_predictions
    )
    print("\n" + "=" * 80)
    print("STRATEGY COMPLETE")
    print("=" * 80)
    print("\nFOLD SUMMARY:")
    print(
        f"{'Fold':<6} {'IS Sharpe':<10} {'OOS Sharpe':<11} {'WFE':<6} {'DSR':<6} {'GRI':<6} {'Status'}"
    )
    print("-" * 80)
    for fold in detector.fold_metrics:
        status = "âœ“" if fold.gri > 0.65 else "âš " if fold.gri > 0.5 else "âœ—"
        print(
            f"{fold.fold_num:<6} {fold.sharpe_in_sample:<10.2f} {fold.sharpe_out_of_sample:<11.2f} {fold.wfe:<6.2f} {fold.dsr:<6.2f} {fold.gri:<6.2f} {status}"
        )
    print("\nSTEP 6: GENERATING VISUALIZATIONS")
    try:
        benchmark_data = data_dict.get(config.BENCHMARK_TICKER, None)
        visualizer = StrategyVisualizer(results, config, detector, benchmark_data)
        fig = visualizer.create_full_dashboard()
        output_path = "results/dcc-garch-consensus/strategy_dashboard.html"
        fig.write_html(output_path)
        print(f"âœ“ Dashboard saved to: {output_path}")
    except Exception as e:
        print(f"âš  Visualization error: {str(e)}")
        print("  Continuing without visualizations...")
    print("\nSTEP 7: MONTE CARLO SIMULATION")
    try:
        simulator = MonteCarloSimulator(
            portfolio_history=results["portfolio_history"], n_simulations=10000, forward_days=63
        )
        simulator.run_simulation()
        mc_fig = simulator.create_visualization()
        mc_path = "results/dcc-garch-consensus/monte_carlo_analysis.html"
        mc_fig.write_html(mc_path)
        print(f"âœ“ Monte Carlo saved to: {mc_path}")
        stats = simulator.calculate_statistics()
        print("\nMONTE CARLO SUMMARY:")
        print(
            f"  Expected Return: {stats['mean_return']:.2f}% (Median: {stats['median_return']:.2f}%)"
        )
        print(f"  VaR 95%: {stats['var_95']:.2f}% | CVaR 95%: {stats['cvar_95']:.2f}%")
        print(f"  Probability of Profit: {stats['prob_profit']:.1f}%")
        print(f"  Expected Max Drawdown: {stats['mean_max_dd']:.2f}%")
    except Exception as e:
        print(f"âš  Monte Carlo error: {str(e)}")
        print("  Continuing without Monte Carlo...")
    return results


if __name__ == "__main__":
    results = main()
